# UB disengagement reason categorization

Classifies the free-text note in **`Reason for Disengagement:`** (the column *with* the colon) of the PB 4108 disengagement export, for tickets dated **on or before 31 Mar 2024** (7,860 tickets). Tickets from April 2024 onwards are out of scope: the form's structured fields are used for those instead.

Every ticket gets three levels: **`main_category`** (5 groups: Service Dissatisfaction, Resignation, Business Reason, For Record, Other/Unknown), **`primary_reason_id`** (the approved taxonomy category, the sub-category) and **`detail_category`** (one level below that). The main and detail maps are in CONFIG (`main_category_map`, `detail_category_map`).

The pipeline: **discover first, classify second.**

| Phase | What it does | Checkpoint (in `artifacts/`) |
|---|---|---|
| 0 | Config, logging, helpers | `run.log` |
| 1 | Load, profile, EDA (n-grams, note types by year, form-value conflicts) | `01_profile.json`, `01_loaded.pkl` |
| 2 | Conservative cleaning in new columns, BLANK vs VALID_TEXT | `02_clean.pkl` |
| 3 | Sentence embeddings, one per unique cleaned note, cached | `embeddings/<model>.npz` |
| 4 | BERTopic (UMAP + HDBSCAN) across several `min_cluster_size` values | `04_clusters_mcs<N>.parquet` |
| 5 | Cluster summaries and a separate outlier (-1) analysis | `05_cluster_cards.json` |
| 6 | LLM taxonomy discovery, then consolidation → proposed taxonomy for **human review** | `llm/`, `taxonomy/` |
| 7 | Per-ticket classification against the **approved** taxonomy; **7b** fills notes that give no reason from the form fields | `llm/classification/` |
| 8 | Validation sample and metrics; **8b** blind spot-check; **8c** gold dataset; **8d** blind spot-check of form-derived labels | `outputs/` |
| 9 | Excel workbook | `outputs/` |
| 10 | Categorized copy of the source workbook (values only) | next to the input file |
| 11 | Backfill of the form structure (Vol/Inv, UB Type, Master Reason) for tickets to Mar-2024 | next to the input file |

**Outputs.** In `outputs/` (the repo, gitignored):
* `ub_reason_classification.xlsx`;
* the validation, spot-check and gold files.

Next to the input file (OneDrive Data folder):
* *`<input> - categorized.xlsx`*: the whole source workbook, values only, with `main_cat` / `sub_cat` / `detail_cat` after `Ticket no.`;
* *`<input> - backfill to 2024-03.xlsx`*: one row per in-scope ticket with Vol/Inv, UB Type and Master Reason backfilled, plus a *Category definitions* sheet.

Open questions for the reviewer are kept in *UB reason categorization - open questions.xlsx* in the same folder.

**LLM backend.** No API key is used. The LLM steps run through a *file exchange*: the notebook writes a request file (prompt + payload) to `artifacts/llm/`, Claude (in the Claude Code session) writes the matching response file, and the notebook validates it against a JSON schema. To switch to an API later, replace `llm_exchange()` in Phase 0. Nothing else changes.

**Privacy.** Notes name staff and clients, so everything except this notebook is gitignored (see `.gitignore`). The notebook's printed outputs also show notes.

## Status and open questions (25 Nov 2026)

**Status.** Taxonomy `v1-c29284b4` (18 categories, all approved) is the current version.
* All 2,101 unique notes are classified from the note text: 3,998 tickets.
* Step 7b fills 3,848 of the 3,865 no-reason tickets from the form fields.
* Rules handle 19 more: 12 blank notes, 4 test tickets, 3 cancelled tickets.

27 tickets end in Other/Unknown: 16 BLANK, 6 DELETED, 5 UNKNOWN.

| Main category | Tickets |
|---|---|
| Resignation | 2,782 |
| Business Reason | 2,054 |
| Service Dissatisfaction | 1,759 |
| For Record (Master Reason: *For Record Purpose Only*) | 1,238 |
| Other/Unknown | 27 |

**Checks**

| Check | Size | Result |
|---|---|---|
| Validation sample (stratified; used to write the rules) | 499 notes | 87.8% exact, 91.4% reviewer rules, 95% main category |
| Blind spot-check (fresh random notes, 8b) | 100 notes | 92% exact, 100% main category |
| Blind spot-check of form-derived labels (8d) | 50 tickets (41 labelled) | 95% main category; 52% sub-category exact, 81% counting a general Business Reason as agreeing |
| Blind Master Reason check (5 reviewer agents, all evidence) | 500 random tickets | 97.2% (486 of 500); 95.4% before the 25 Nov fixes. The sheet's own Master Reason scores 87.4% |

v1 scored 91.8% under the reviewer rules against v2's 91.4%, a tie, so no v3 is planned. The gold dataset holds 600 human-labelled notes.

**Open questions.** The full list, with context, options and a column for answers, is in *UB reason categorization - open questions.xlsx* (Data folder). In short:
1. **Main categories.** Confirm the five names (copied from a cut-off screenshot) and the one-line definitions.
2. **Vol/Inv for Cloudstaff-initiated medical exits.** 13 tickets come out Voluntary because STAFF_MEDICAL rolls up to Resignation, but the form tree has them under Involuntary.
3. **Form-tree judgement calls.** 20 of them, 8 of which touch tickets. The big two are Company Restructure → ROLE_NO_LONGER_NEEDED (162) and client code of conduct → MISCONDUCT (107).
4. **Form spot-check follow-ups.** 8 tickets where the free text and the form disagree.
5. **Read the comments before the form fields?** 2,121 form-labelled tickets have comment text. 7 of the 14 remaining Master Reason misses need it. The other 7 are genuinely ambiguous.
6. Smaller items:
   * whether to trust the form's "Client Pull-out";
   * whether "nearest" UB Type_backfill values are good enough;
   * Master Reason_backfill (= main_cat) vs the sheet's Future Re-Hire rule (639 tickets differ);
   * a "with replacement" flag;
   * the old form-vs-note discrepancy file.

**Decided on 25 Nov 2026.** The questions file has the full list on its *Decided* sheet.
* **Note wins.** The note's reason always wins; the form fields are used only when the note gives no reason. April 2024 onwards stays out of scope.
* **Detail categories.** The unused detail categories (`staff_deceased`, `misconduct_data_security`, `performance_productivity`) are dropped. STAFF_MEDICAL merges into `resignation_health`.
* **Pricing** stays a SERVICE_DISSATISFACTION subcategory (13 tickets).
* **Non-payment.** "client has gone unresponsive" in UB Main Reason → NON_PAYMENT (34 tickets).
* **Fixes after the Master Reason check** (42 tickets changed):
  * a buy-out in any spelling;
  * four more Disengagement Category values: Client Terminated, Business downturn - Budget Issues, Suspended Account, CS Now reliever;
  * test tickets → BLANK;
  * cancelled tickets → DELETED.
* **Backfill.** UB Type_backfill: the note wins, then the ticket's own form value, then the nearest tree value, flagged in the basis column. Master Reason_backfill = main_cat, with For Record written *For Record Purpose Only*. Vol/Inv_backfilled comes from main_cat.
* **Phases 10 and 11** build the categorized copy and the backfill file on every run.
* **Raw file.** The new raw file (25 Nov 13:49) has identical in-scope rows.


In [ ]:
import os, re, json, html, hashlib, unicodedata, logging, datetime as dt, warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

NB_DIR = Path.cwd()
if not (NB_DIR / "ub_reason_categorization.ipynb").exists():
    NB_DIR = Path(r"C:\Users\Alfonso(FonsoP)Panli\repos\cloudstaff_dev\reference_code\UB Reason categorizing")

CONFIG = {
    # --- input ---------------------------------------------------------------
    "input_path": r"C:\Users\Alfonso(FonsoP)Panli\OneDrive - Cloudstaff Pty Ltd\Desktop\Projects\Churn Analysis\Data\PB4108 Disengagement 2020 onwards.xlsx",
    "input_type": "excel",              # "excel" or "csv"
    "sheet_name": "PB 4108",
    "id_col": "Ticket no.",
    "text_col": "Reason for Disengagement:",   # free-text note (WITH colon); the no-colon column is the form dropdown
    "date_col": "Date Disengagement",
    "date_max": "2024-03-31",           # inclusive; None = no filter
    "duplicate_id_handling": "suffix",  # "suffix" (keep all, add __2, __3) | "keep_first" | "error"
    "form_values": ["Voluntary", "Involuntary", "For Record Purpose Only", "Deleted"],  # dropdown values that show up as notes
    "short_note_words": 2,              # notes with <= this many words count as "very short"
    # --- outputs -------------------------------------------------------------
    "artifact_dir": str(NB_DIR / "artifacts"),
    "taxonomy_dir": str(NB_DIR / "taxonomy"),
    "output_dir": str(NB_DIR / "outputs"),
    "output_workbook": "ub_reason_classification.xlsx",
    # --- embeddings ----------------------------------------------------------
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_batch_size": 64,
    # --- clustering ----------------------------------------------------------
    "umap": dict(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42),
    "hdbscan": dict(min_samples=5, metric="euclidean", cluster_selection_method="eom", prediction_data=True),
    "min_cluster_sizes": [10, 20, 30, 50],
    "selected_min_cluster_size": 10,    # chosen after reviewing the comparison in Phase 4
    "outlier_group_distance": 0.35,     # cosine distance for grouping outliers into mini-themes
    # --- LLM -----------------------------------------------------------------
    "llm_backend": "file_exchange",     # Claude in the Claude Code session writes the response files
    "llm_model": "claude-opus-5-5 (in-session, file exchange)",
    "taxonomy_prompt_version": "tax-v1",
    "classification_prompt_version": "cls-v1",
    "classification_batch_size": 150,
    "confidence_threshold": 0.75,
    "validation_sample_size": 500,
    # reviewer scoring rules (validation "Accuracy (reviewer rules)"): agreed with the reviewer 2026-11-25
    "validation_equivalent_pairs": [["ATTENDANCE_AWOL", "SERVICE_DISSATISFACTION"]],  # both true: right if one is primary and the other secondary
    "validation_rollup": {"ROLE_NO_LONGER_NEEDED": "BUSINESS_REASON"},                  # counted as business reason at large
    "spotcheck_size": 100,
    "form_spotcheck_size": 50,          # 8d: blind check of tickets labelled from the form fields (step 7b)
    "categorized_copy": True,           # Phase 10: write "<input> - categorized.xlsx" next to the input (Windows + Excel only)
    # reviewer's main categories (2026-11-25): every approved category_id must appear here
    "main_category_map": {
        "SERVICE_DISSATISFACTION": "Service Dissatisfaction", "ATTENDANCE_AWOL": "Service Dissatisfaction", "MISCONDUCT": "Service Dissatisfaction",
        "RESIGNATION": "Resignation", "STAFF_MEDICAL": "Resignation",
        "BUSINESS_REASON": "Business Reason", "BUSINESS_DOWNTURN": "Business Reason", "ROLE_NO_LONGER_NEEDED": "Business Reason",
        "ROLE_MOVED_ONSHORE_OR_PROVIDER": "Business Reason", "CLIENT_CLOSURE_OR_ACQUISITION": "Business Reason",
        "CLIENT_ACCOUNT_EXIT": "Business Reason", "CLIENT_BUYOUT": "Business Reason", "NON_PAYMENT": "Business Reason",
        "FOR_RECORD_PURPOSE_ONLY": "For Record", "FORM_DIRECTION_ONLY": "For Record",
        "BLANK": "Other/Unknown", "DELETED": "Other/Unknown", "UNKNOWN": "Other/Unknown",
    },
    # third level under sub_cat (2026-11-25): category_id -> {model subcategory: detail name}; "" = no subcategory given.
    # STAFF_MEDICAL is merged into resignation_health on the reviewer's decision.
    "detail_category_map": {
        "RESIGNATION": {"": "resignation_general", "Personal (undisclosed)": "resignation_personal_reasons",
                        "Better opportunity / compensation": "resignation_better_opportunity", "Career growth / studies": "resignation_career_studies",
                        "Health / medical": "resignation_health", "Family / caring": "resignation_family", "Relocation / overseas": "resignation_relocation",
                        "Return-to-office / work arrangement": "resignation_work_arrangement", "Own business": "resignation_own_business",
                        "Schedule / night shift": "resignation_schedule", "Dissatisfied with Cloudstaff / client / team": "resignation_dissatisfied"},
        "STAFF_MEDICAL": {"": "resignation_health", "Unfit to work / prolonged sick leave": "resignation_health",
                          "Cannot work night shift (health)": "resignation_health", "Deceased": "resignation_health"},
        "SERVICE_DISSATISFACTION": {"": "performance_general", "Performance / quality / productivity": "performance_quality",
                                    "Performance - insufficient documentation": "performance_undocumented", "Skills or role fit": "performance_skills_role_fit",
                                    "Failed training / onboarding": "performance_failed_training", "Failed / declined PIP": "performance_failed_pip",
                                    "Communication": "performance_communication", "Cloudstaff service / cost / value": "cloudstaff_service_cost_value"},
        "ATTENDANCE_AWOL": {"": "attendance_general", "Absenteeism / tardiness": "attendance_absenteeism_tardiness", "NCNS / AWOL": "attendance_ncns_awol",
                            "Failed RTWO": "attendance_failed_rtwo", "No-show after onboarding": "attendance_no_show_onboarding",
                            "Connectivity / power outages": "attendance_connectivity"},
        "MISCONDUCT": {"": "misconduct_general", "Call avoidance": "misconduct_call_avoidance", "Fraud / falsification": "misconduct_fraud_falsification",
                       "Code of conduct / behaviour": "misconduct_code_of_conduct"},
        "BUSINESS_REASON": {"": "business_reason_general", "Unspecified business reason": "business_reason_general",
                            "Client decided not to continue": "business_client_decided_not_to_continue",
                            "Requirements / plans changed": "business_requirements_changed", "Business & structural changes": "business_structural_changes"},
        "BUSINESS_DOWNTURN": {"": "downturn_general", "Slowdown / reduced volume": "downturn_slowdown_volume", "COVID / lockdown": "downturn_covid",
                              "Seasonal": "downturn_seasonal", "Budget / cost cutting": "downturn_budget_cost_cutting"},
        "ROLE_NO_LONGER_NEEDED": {"": "role_no_longer_needed_general", "Role eliminated": "role_eliminated", "Restructure / reorganisation": "role_restructure",
                                  "Automation / process change": "role_automation", "Project / temp role ended": "role_project_ended"},
        "ROLE_MOVED_ONSHORE_OR_PROVIDER": {"": "role_moved_general", "Moved onshore / local hire": "role_moved_onshore", "Moved in-house": "role_moved_in_house",
                                           "Moved to cheaper provider / competitor": "role_moved_provider"},
        "CLIENT_CLOSURE_OR_ACQUISITION": {"": "client_closure_general", "Closure / dissolution": "client_closure", "Sold / acquired / merged": "client_sold_acquired"},
        "CLIENT_ACCOUNT_EXIT": {"": "account_exit_general", "Contract terminated": "account_contract_terminated", "Pull-out": "account_pull_out",
                                "Sub-client pull-out": "account_sub_client_pull_out", "Breach of contract": "account_breach_of_contract"},
        "NON_PAYMENT": {"": "non_payment", "Non-payment": "non_payment"},
        "CLIENT_BUYOUT": {"": "buyout", "Buy-out": "buyout", "Direct hire": "buyout_direct_hire"},
        "FOR_RECORD_PURPOSE_ONLY": {"": "record_general", "Form value": "record_form_value",
                                    "Work-arrangement change (OfficeFlex / SOF / WFH)": "record_work_arrangement_change",
                                    "Entity / billing change": "record_entity_billing_change", "Engagement-type change": "record_engagement_type_change",
                                    "Internal transfer": "record_internal_transfer"},
        "FORM_DIRECTION_ONLY": {"": "form_direction_general", "Voluntary (form value)": "form_voluntary", "Involuntary (form value)": "form_involuntary"},
        "DELETED": {"": "deleted", "Deleted": "deleted", "Cancelled / withdrawn": "deleted_cancelled_withdrawn"},
        "BLANK": {"": "blank"},
        "UNKNOWN": {"": "unknown"},
    },              # fresh blind sample, notes never in any earlier validation file
    "random_seed": 42,
}

ART = Path(CONFIG["artifact_dir"]); TAX = Path(CONFIG["taxonomy_dir"]); OUT = Path(CONFIG["output_dir"])
LLM_DIR = ART / "llm"
for p in [ART, TAX, OUT, LLM_DIR, ART / "embeddings", LLM_DIR / "classification"]:
    p.mkdir(parents=True, exist_ok=True)

# logging: file + notebook, never secrets
logger = logging.getLogger("ub_reason")
logger.setLevel(logging.INFO)
if not logger.handlers:
    fh = logging.FileHandler(ART / "run.log", encoding="utf-8")
    fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
    logger.addHandler(fh)
RUN = {"started": dt.datetime.now().isoformat(timespec="seconds"), "warnings": [], "errors": [], "status": {}}
def warn(msg):
    RUN["warnings"].append(msg); logger.warning(msg); print("WARNING:", msg)
logger.info("Run started")
print("Notebook dir:", NB_DIR)

In [ ]:
# ---- LLM file exchange --------------------------------------------------------
# request:  artifacts/llm/<name>.request.json   (prompt + payload, written by the notebook)
# response: artifacts/llm/<name>.response.json  (written by the LLM, i.e. Claude in-session)
# Swap this function for an API call to automate; the rest of the notebook only sees the parsed JSON.
import jsonschema

class LLMResponsePending(Exception):
    pass

def llm_exchange(name, prompt, payload, schema=None, subdir=None):
    d = LLM_DIR / subdir if subdir else LLM_DIR
    d.mkdir(parents=True, exist_ok=True)
    req = d / f"{name}.request.json"; resp = d / f"{name}.response.json"
    request = {"name": name, "model": CONFIG["llm_model"], "prompt": prompt, "payload": payload}
    req.write_text(json.dumps(request, indent=1, ensure_ascii=False), encoding="utf-8")
    if not resp.exists():
        logger.info(f"LLM request written, awaiting response: {req.name}")
        raise LLMResponsePending(f"Waiting for {resp}")
    try:
        data = json.loads(resp.read_text(encoding="utf-8"))
    except json.JSONDecodeError as e:
        RUN["errors"].append(f"{resp.name}: invalid JSON ({e})"); logger.error(f"invalid JSON {resp.name}")
        raise
    if schema is not None:
        jsonschema.validate(data, schema)   # raises on invalid; never silently accepted
    logger.info(f"LLM response loaded: {resp.name}")
    return data

def sha(s, n=16):
    return hashlib.sha1(str(s).encode("utf-8")).hexdigest()[:n]

## Phase 1 · Load and profile

In [ ]:
def load_input(cfg):
    p = Path(cfg["input_path"])
    if not p.exists():
        raise FileNotFoundError(f"Input not found: {p}")
    if cfg["input_type"] == "excel":
        try:
            df = pd.read_excel(p, sheet_name=cfg["sheet_name"])
        except PermissionError:                        # Excel has the file open: a file copy still works
            import shutil
            tmp = Path(cfg["artifact_dir"]) / f"_input_copy{p.suffix}"
            shutil.copy2(p, tmp)
            warn(f"{p.name} is locked (open in Excel?) - reading a copy made at {dt.datetime.now():%H:%M}")
            df = pd.read_excel(tmp, sheet_name=cfg["sheet_name"])
    elif cfg["input_type"] == "csv":
        df = pd.read_csv(p)
    else:
        raise ValueError("input_type must be 'excel' or 'csv'")
    missing = [c for c in (cfg["id_col"], cfg["text_col"]) if c not in df.columns]
    if missing:
        raise KeyError(f"Configured columns missing from input: {missing}")
    df.insert(0, "source_row", df.index + 2)          # Excel row number in the source sheet
    if cfg.get("date_max"):
        dates = pd.to_datetime(df[cfg["date_col"]], errors="coerce")
        n_bad = dates.isna().sum()
        if n_bad: warn(f"{n_bad} rows have an unparseable {cfg['date_col']} and are excluded by the date filter")
        df = df[dates <= pd.Timestamp(cfg["date_max"])].copy()
    return df.reset_index(drop=True)

def assign_record_ids(df, cfg):
    ids = df[cfg["id_col"]].astype("string")
    if ids.isna().any():
        warn(f"{ids.isna().sum()} rows have no {cfg['id_col']}; given ROW-<source_row> ids")
        ids = ids.fillna("ROW-" + df["source_row"].astype(str))
    dup = ids.duplicated(keep=False)
    if dup.any():
        warn(f"{dup.sum()} rows share a duplicated {cfg['id_col']}: {sorted(ids[dup].unique())}")
        if cfg["duplicate_id_handling"] == "error":
            raise ValueError("Duplicate record IDs")
        if cfg["duplicate_id_handling"] == "keep_first":
            keep = ~ids.duplicated(keep="first"); df, ids = df[keep].copy(), ids[keep]
        else:
            n = ids.groupby(ids).cumcount() + 1
            ids = ids.where(n == 1, ids + "__" + n.astype(str))
    df.insert(0, "record_id", ids.values)
    return df, dup

df = load_input(CONFIG)
df, dup_id_mask = assign_record_ids(df, CONFIG)
TEXT = CONFIG["text_col"]
print(f"Loaded {len(df):,} rows x {df.shape[1]} cols ({CONFIG['date_col']} <= {CONFIG['date_max']})")
display(df.loc[dup_id_mask, ["record_id", CONFIG["id_col"], "Engagement ID", "Client ShortCode", CONFIG["date_col"], TEXT]])

In [ ]:
def profile_notes(df, text_col, short_words):
    raw = df[text_col]
    stripped = raw.astype("string").str.strip()
    is_blank = stripped.isna() | (stripped == "")
    nb = stripped[~is_blank]
    words = nb.str.split().str.len()
    vc = nb.value_counts()
    n = len(df)
    rows = [
        ("Total records", n),
        ("Blank notes", int(is_blank.sum())),
        ("Non-blank notes", int((~is_blank).sum())),
        ("Unique non-blank notes", int(nb.nunique())),
        ("Records whose note is repeated elsewhere", int(nb.duplicated(keep=False).sum())),
        ("Records with a note seen only once", int((vc == 1).sum())),
        (f"Very short notes (<= {short_words} words)", int((words <= short_words).sum())),
        ("Notes over 50 words", int((words > 50).sum())),
    ]
    prof = pd.DataFrame(rows, columns=["metric", "records"])
    prof["pct_of_all_records"] = (prof["records"] / n * 100).round(1)
    length = words.describe(percentiles=[.25, .5, .75, .9, .99]).round(1)
    longest = (df.loc[~is_blank, ["record_id", text_col]].assign(words=words)
               .sort_values("words", ascending=False).head(10))
    top = vc.head(25).rename_axis("raw_note").reset_index(name="records")
    top["pct_of_all_records"] = (top["records"] / n * 100).round(1)
    return prof, length, longest, top

profile, note_length, longest_notes, top_raw_notes = profile_notes(df, TEXT, CONFIG["short_note_words"])
display(profile)
print("Words per non-blank note:"); display(note_length.to_frame("words").T)
display(top_raw_notes)

In [ ]:
print("Longest notes (often pasted client emails):")
display(longest_notes.assign(preview=longest_notes[TEXT].str.slice(0, 160))[["record_id", "words", "preview"]])

(ART / "01_profile.json").write_text(json.dumps({
    "profile": profile.to_dict("records"), "note_length": note_length.to_dict(),
    "top_raw_notes": top_raw_notes.to_dict("records"),
    "duplicate_ids": df.loc[dup_id_mask, "record_id"].tolist()}, indent=1, default=str), encoding="utf-8")
df.to_pickle(ART / "01_loaded.pkl")   # pickle keeps mixed-type source columns exactly
RUN["status"]["phase1"] = "done"; logger.info(f"Phase 1: {len(df)} records loaded")

### EDA 1 · What kind of text is in the note field, by year?

The note field holds three different things: real free text, a **form value** copied in (`Voluntary`, `Involuntary`, `For Record Purpose Only`), or nothing. The mix changes sharply by year, so free-text notes describe 2020–22 much better than 2023–24.

In [ ]:
fv = {v.lower() for v in CONFIG["form_values"]}
s = df[TEXT].astype("string").str.strip()
df["note_kind"] = np.where(s.isna() | (s == ""), "BLANK",
                  np.where(s.str.lower().isin(fv), "FORM VALUE", "FREE TEXT"))
kind_year = pd.crosstab(df["Year Disengagement"], df["note_kind"], margins=True, margins_name="Total")
display(kind_year)
display((pd.crosstab(df["Year Disengagement"], df["note_kind"], normalize="index") * 100).round(0).astype(int).add_suffix(" %"))

### EDA 2 · Most common 2–4-word phrases (free-text notes only)

Counted once per ticket. Form values are left out because they would dominate. Stop words are kept inside phrases ("no longer needed" needs them), but phrases that start or end with a stop word are dropped.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
NEGATIONS = {"no", "not", "never", "nor", "none", "cannot"}
STOP = set(ENGLISH_STOP_WORDS) - NEGATIONS

def top_ngrams(texts, n, k=15):
    cv = CountVectorizer(ngram_range=(n, n), binary=True, lowercase=True, token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z']+\b")
    X = cv.fit_transform(texts)
    counts = pd.Series(np.asarray(X.sum(0)).ravel(), index=cv.get_feature_names_out())
    keep = [g for g in counts.index if g.split()[0] not in STOP and g.split()[-1] not in STOP]
    return counts[keep].sort_values(ascending=False).head(k)

free = df.loc[df["note_kind"] == "FREE TEXT", TEXT].astype(str).str.replace("_x000D_", " ")
ngram_tables = {n: top_ngrams(free, n) for n in (2, 3, 4)}
display(pd.concat({f"{n}-grams": t.rename_axis("phrase").reset_index(name="tickets") for n, t in ngram_tables.items()}, axis=1))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True)
for ax, (n, t) in zip(axes, ngram_tables.items()):
    t = t.iloc[::-1]
    ax.barh(t.index, t.values, color="#3B6FB6", height=0.7)
    ax.set_title(f"Top {n}-word phrases (tickets)", loc="left", fontsize=11)
    ax.spines[["top", "right"]].set_visible(False); ax.tick_params(labelsize=8)
    ax.grid(axis="x", color="#e5e5e5"); ax.set_axisbelow(True)
plt.show()

### EDA 3 · Do the `Voluntary` / `Involuntary` notes agree with the form?

These notes are the direction dropdown copied into the note field (2022+). Checked against the structured fields:
* the dropdown column `Reason for Disengagement` (no colon),
* the sub-reason fields (`Type of Voluntary Reason`, `Type of Involuntary`, `Type of Customer Initiated`, `Reason for Resignation`),
* the workbook's derived `Master Reason`.

The free-text notes are also checked: notes whose *wording* is clearly a resignation, or clearly client / business / performance, against the dropdown direction.

In [ ]:
ncl = df[TEXT].astype("string").str.strip().fillna("<blank>")
grp = ncl.where(ncl.isin(["Voluntary", "Involuntary", "For Record Purpose Only", "<blank>"]), "<free text>")
print("Note value vs dropdown direction"); display(pd.crosstab(grp, df["Reason for Disengagement"], margins=True))
print("Note value vs derived Master Reason"); display(pd.crosstab(grp, df["Master Reason"], margins=True))

vol, inv = df[ncl == "Voluntary"], df[ncl == "Involuntary"]
inv_fields = ["Type of Involuntary", "Type of Customer Initiated", "Type of Cloudstaff Initiated"]
vol_conf = vol[vol[inv_fields].notna().any(axis=1)]
print(f"'Voluntary' notes that also carry involuntary sub-reasons: {len(vol_conf)} of {len(vol)}")
display(pd.crosstab(vol_conf["Type of Customer Initiated"].fillna("(none)"), vol_conf["Master Reason"]))
real_vol_reason = inv["Reason for Resignation"].notna() | (inv["Type of Voluntary Reason"].notna() & ~inv["Type of Voluntary Reason"].astype(str).str.startswith("NA."))
print(f"'Involuntary' notes that carry a resignation reason: {real_vol_reason.sum()} of {len(inv)}")
display(inv.loc[real_vol_reason, "Reason for Resignation"].value_counts(dropna=False).head(8).to_frame("tickets"))

In [ ]:
ft = grp == "<free text>"
note = df[TEXT].astype(str)
resign_words = note.str.contains("resign", case=False) & ~note.str.contains("client|business|pull", case=False)
client_words = (note.str.contains(r"performance|pull ?out|business|role no longer|downturn|slow ?down|ncns|awol|attendance", case=False)
                & ~note.str.contains("resign", case=False))
conf = pd.DataFrame({
    "free-text wording": ["clearly resignation", "clearly client / business / performance"],
    "tickets": [int((ft & resign_words).sum()), int((ft & client_words).sum())],
    "dropdown = Voluntary": [int((ft & resign_words & (df["Reason for Disengagement"] == "Voluntary")).sum()),
                             int((ft & client_words & (df["Reason for Disengagement"] == "Voluntary")).sum())],
    "dropdown = Involuntary": [int((ft & resign_words & (df["Reason for Disengagement"] == "Involuntary")).sum()),
                               int((ft & client_words & (df["Reason for Disengagement"] == "Involuntary")).sum())],
})
display(conf)
print("Examples: client/business/performance wording but dropdown = Voluntary (so Master Reason = Resignation)")
display(df.loc[ft & client_words & (df["Reason for Disengagement"] == "Voluntary"), ["record_id", TEXT, "Master Reason"]]
        .assign(**{TEXT: lambda x: x[TEXT].str.slice(0, 130)}).head(10))

**Takeaways**
* A `Voluntary` / `Involuntary` note carries **direction only, no reason**. It matches the dropdown in almost every case. The disagreements are *inside the structured fields*: 63 `Voluntary` tickets also carry involuntary sub-reasons (mostly *Customer Initiated → Business Reasons / Service Dissatisfaction*), but `Master Reason` still calls them Resignation.
* Free text clearly about the client / business / performance sometimes sits under a `Voluntary` dropdown. Those tickets inherit `Master Reason = Resignation`.
* So the note classification below uses **the note text only**. It is compared with `Master Reason` afterwards (Phase 9) to surface disagreements, instead of copying the form. **Exception:** notes that give no reason (direction only, blank, unknown) are filled from the form fields in step 7b, at the reviewer's request (2026-11-25).

## Phase 2 · Preprocess (conservative)

The original note is untouched. `note_clean` is new. There's no stop-word removal, stemming or lower-casing before embedding, and negations are kept. A note is **BLANK** if it's empty, whitespace only, a template placeholder (`{{FIELD_…}}`) or has no letters or digits. Everything else is **VALID_TEXT**, including junk like "test", which the classifier can still call BLANK.

In [ ]:
HTML_TAG = re.compile(r"</?[a-zA-Z][a-zA-Z0-9]*(\s[^<>]*)?/?>")
PLACEHOLDER = re.compile(r"^\{\{[^{}]*\}\}$")
QUOTES = {"\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"', "\u2013": "-", "\u2014": "-", "\u00a0": " "}

def clean_note(x):
    if x is None or (not isinstance(x, str) and pd.isna(x)):
        return ""
    s = str(x).replace("_x000D_", " ")
    s = html.unescape(s)
    s = HTML_TAG.sub(" ", s)
    s = unicodedata.normalize("NFKC", s)
    for a, b in QUOTES.items():
        s = s.replace(a, b)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def processing_status(clean):
    if clean == "" or PLACEHOLDER.match(clean) or not re.search(r"[A-Za-z0-9]", clean):
        return "BLANK"
    return "VALID_TEXT"

df["note_clean"] = df[TEXT].map(clean_note)
df["processing_status"] = df["note_clean"].map(processing_status)
df["note_key"] = np.where(df["processing_status"] == "VALID_TEXT", df["note_clean"].map(sha), "")
df.to_pickle(ART / "02_clean.pkl")
display(df["processing_status"].value_counts().to_frame("records"))
print("Cleaned notes that became BLANK although the raw cell had text:")
display(df.loc[(df["processing_status"] == "BLANK") & df[TEXT].notna(), TEXT].value_counts().to_frame("records"))
RUN["status"]["phase2"] = "done"

## Phase 3 · Sentence embeddings

One embedding per **unique cleaned note**. `unique_notes` maps each note back to its records (`records` = how many tickets share it). Embeddings are cached in `artifacts/embeddings/` keyed by note hash, so reruns only embed new notes.

In [ ]:
from sentence_transformers import SentenceTransformer

valid = df[df["processing_status"] == "VALID_TEXT"]
unique_notes = (valid.groupby("note_key")
    .agg(note_clean=("note_clean", "first"), records=("record_id", "size"),
         note_original=(TEXT, lambda s: s.value_counts().index[0]))   # most common raw wording
    .sort_values("records", ascending=False).reset_index())
assert unique_notes["records"].sum() == len(valid)

def get_embeddings(texts, keys, model_name):
    cache = ART / "embeddings" / (model_name.replace("/", "__") + ".npz")
    have = {}
    if cache.exists():
        z = np.load(cache, allow_pickle=False)
        have = dict(zip(z["keys"].tolist(), z["vectors"]))
    todo = [(k, t) for k, t in zip(keys, texts) if k not in have]
    if todo:
        model = SentenceTransformer(model_name)
        vecs = model.encode([t for _, t in todo], batch_size=CONFIG["embedding_batch_size"],
                            normalize_embeddings=True, show_progress_bar=False)
        have.update({k: v for (k, _), v in zip(todo, vecs)})
        np.savez(cache, keys=np.array(list(have.keys())), vectors=np.vstack(list(have.values())))
    logger.info(f"Embeddings: {len(keys)} needed, {len(todo)} newly computed ({model_name})")
    print(f"{len(keys):,} unique notes; {len(todo):,} newly embedded, {len(keys)-len(todo):,} from cache")
    return np.vstack([have[k] for k in keys])

E = get_embeddings(unique_notes["note_clean"].tolist(), unique_notes["note_key"].tolist(), CONFIG["embedding_model"])
print("Embedding matrix:", E.shape)
RUN["status"]["phase3"] = "done"

## Phase 4 · Semantic clusters (BERTopic = UMAP → HDBSCAN → c-TF-IDF keywords)

Clusters are fitted on unique notes. **Sizes are reported in tickets** (the `records` each note represents). Noise stays as cluster **-1** and is never forced into a cluster. UMAP has a fixed seed, so runs repeat.

Note: HDBSCAN can't weight points, so a note repeated 1,000 times counts once when the clusters form.

In [ ]:
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.metrics import silhouette_score

KEYWORD_STOP = sorted(STOP)   # keyword extraction only; embeddings saw the full text

def run_bertopic(docs, emb, min_cluster_size):
    model = BERTopic(
        embedding_model=None,
        umap_model=UMAP(**CONFIG["umap"]),
        hdbscan_model=HDBSCAN(min_cluster_size=min_cluster_size, **CONFIG["hdbscan"]),
        vectorizer_model=CountVectorizer(ngram_range=(1, 3), stop_words=KEYWORD_STOP, min_df=2),
        calculate_probabilities=False, verbose=False)
    topics, probs = model.fit_transform(docs, embeddings=emb)
    return model, np.asarray(topics), np.asarray(probs)

def cluster_quality(labels, emb, weights):
    rows = []
    for c in sorted(set(labels) - {-1}):
        m = labels == c
        cen = emb[m].mean(0); cen /= np.linalg.norm(cen)
        rows.append((c, m.sum(), weights[m].sum(), float((emb[m] @ cen).mean())))
    return pd.DataFrame(rows, columns=["cluster", "unique", "records", "coherence"])

docs = unique_notes["note_clean"].tolist(); w = unique_notes["records"].to_numpy()
experiments, models = [], {}
for mcs in CONFIG["min_cluster_sizes"]:
    path = ART / f"04_clusters_mcs{mcs}.parquet"
    model, topics, probs = run_bertopic(docs, E, mcs)
    models[mcs] = model
    pd.DataFrame({"note_key": unique_notes["note_key"], "cluster_id": topics, "probability": probs}).to_parquet(path, index=False)
    q = cluster_quality(topics, E, w); noise = topics == -1
    sil = silhouette_score(E[~noise], topics[~noise], metric="cosine") if (~noise).sum() > 2 and len(q) > 1 else np.nan
    experiments.append({
        "min_cluster_size": mcs, "clusters": len(q),
        "outlier_unique_%": round(noise.mean() * 100, 1),
        "outlier_records_%": round(w[noise].sum() / w.sum() * 100, 1),
        "largest_cluster_records": int(q["records"].max()), "median_cluster_records": int(q["records"].median()),
        "mean_coherence": round(q["coherence"].mean(), 3), "silhouette_cosine": round(sil, 3)})
    logger.info(f"BERTopic mcs={mcs}: {len(q)} clusters, {noise.sum()} outlier notes")
experiments = pd.DataFrame(experiments)
display(experiments)

In [ ]:
# Side by side: what each configuration calls its biggest clusters (top 4 keywords), to check merging and splitting by eye
def top_clusters(mcs, k=12):
    lab = pd.read_parquet(ART / f"04_clusters_mcs{mcs}.parquet")["cluster_id"].to_numpy()
    q = cluster_quality(lab, E, w).sort_values("records", ascending=False).head(k)
    kw = [", ".join(t for t, _ in models[mcs].get_topic(c)[:4]) for c in q["cluster"]]
    return pd.Series([f"{r} | {k}" for r, k in zip(q["records"], kw)], name=f"mcs={mcs}")
display(pd.concat([top_clusters(m).reset_index(drop=True) for m in CONFIG["min_cluster_sizes"]], axis=1))

**Choosing a configuration.** The choice isn't made by any single metric. Larger `min_cluster_size` merges distinct reasons into a few big clusters, for example resignation sub-reasons, office/WFH moves and business slowdowns. The smallest setting keeps those reasons apart while the clusters stay coherent, and its outliers are reviewed separately in Phase 5 anyway. **`selected_min_cluster_size` is set in CONFIG.**

In [ ]:
SEL = CONFIG["selected_min_cluster_size"]
assign = pd.read_parquet(ART / f"04_clusters_mcs{SEL}.parquet")
unique_notes = unique_notes.drop(columns=["cluster_id", "probability"], errors="ignore").merge(assign, on="note_key", how="left")
topic_model = models[SEL]
labels = unique_notes["cluster_id"].to_numpy()
df = df.drop(columns=["cluster_id"], errors="ignore").merge(unique_notes[["note_key", "cluster_id"]], on="note_key", how="left")
assert len(df) == len(valid) + (df["processing_status"] == "BLANK").sum()
print(f"Selected min_cluster_size={SEL}: {len(set(labels)) - 1} clusters; outliers = {(labels == -1).sum()} notes / {w[labels == -1].sum()} tickets")
RUN["status"]["phase4"] = f"done (mcs={SEL})"

## Phase 5 · Cluster summaries and outlier analysis

For each cluster, the summary shows sizes, c-TF-IDF keywords, mean within-cluster similarity and mean HDBSCAN membership probability. It also lists:
* **representative notes**: chosen for diversity (maximal marginal relevance), starting from the most common note;
* the notes **closest to** and **furthest from** the centroid.

Notes are shown in their original wording.

Outliers (-1) are grouped into mini-themes by cosine distance, so small recurring reasons aren't lost.

In [ ]:
def mmr_pick(idx, emb, weights, k=8, lam=0.6):
    idx = list(idx)
    if len(idx) <= k: return idx
    cen = emb[idx].mean(0)
    rel = emb[idx] @ cen + 0.02 * np.log1p(weights[idx])
    chosen = [idx[int(np.argmax(weights[idx]))]]
    while len(chosen) < k:
        best, best_s = None, -1e9
        for j, i in enumerate(idx):
            if i in chosen: continue
            s = lam * rel[j] - (1 - lam) * max(float(emb[i] @ emb[c]) for c in chosen)
            if s > best_s: best, best_s = i, s
        chosen.append(best)
    return chosen

def fmt(i):
    r = unique_notes.iloc[i]
    return f"[{r.records}] {r.note_original.replace('_x000D_', ' ').strip()[:300]}"

def cluster_card(cid, idx, kind="cluster", keywords=None):
    idx = np.asarray(idx)
    sub = E[idx]; cen = sub.mean(0); cen /= np.linalg.norm(cen)
    sim = sub @ cen; order = idx[np.argsort(-sim)]
    pair = (sub @ sub.T)[np.triu_indices(len(idx), 1)] if len(idx) > 1 else np.array([1.0])
    probs = unique_notes.loc[idx, "probability"]
    return {
        "cluster_id": str(cid), "kind": kind,
        "unique_notes": int(len(idx)), "records": int(w[idx].sum()),
        "keywords": keywords or [],
        "mean_within_similarity": round(float(pair.mean()), 3),
        "mean_membership_probability": round(float(probs.mean()), 3) if kind == "cluster" else None,
        "representative_notes": [fmt(i) for i in mmr_pick(idx, E, w)],
        "closest_to_centroid": [fmt(i) for i in order[:5]],
        "furthest_from_centroid": [fmt(i) for i in order[::-1][:4]] if len(idx) > 5 else [],
    }

cards = []
for c in sorted(set(labels) - {-1}):
    idx = np.where(labels == c)[0]
    kw = [t for t, _ in topic_model.get_topic(c)[:10]]
    cards.append(cluster_card(f"C{c:02d}", idx, "cluster", kw))
cards.sort(key=lambda d: -d["records"])
print(f"{len(cards)} clusters summarised")

In [ ]:
# ---- Outliers (-1): group into mini-themes, flag data-quality issues ----
from sklearn.cluster import AgglomerativeClustering
out_idx = np.where(labels == -1)[0]
og = AgglomerativeClustering(n_clusters=None, metric="cosine", linkage="average",
                             distance_threshold=CONFIG["outlier_group_distance"]).fit(E[out_idx]).labels_
unique_notes["outlier_group"] = pd.NA
unique_notes.loc[out_idx, "outlier_group"] = og

ctf = CountVectorizer(ngram_range=(1, 2), stop_words=KEYWORD_STOP, min_df=1)
outlier_cards, singles = [], []
for g in pd.Series(og).value_counts().index:
    idx = out_idx[og == g]
    if len(idx) >= 3 or w[idx].sum() >= 5:
        try:
            X = ctf.fit_transform(unique_notes.loc[idx, "note_clean"])
            kw = list(pd.Series(np.asarray(X.multiply(w[idx][:, None]).sum(0)).ravel(), index=ctf.get_feature_names_out()).nlargest(8).index)
        except ValueError:      # group made only of stop words, e.g. "Other"
            kw = []
        outlier_cards.append(cluster_card(f"O{len(outlier_cards):02d}", idx, "outlier_group", kw))
    else:
        singles.extend(idx.tolist())
outlier_cards.sort(key=lambda d: -d["records"])

sing = unique_notes.loc[singles]
dq = pd.DataFrame({
    "issue": ["pasted email / long text (>50 words)", "contains a URL only or mostly", "test / keyboard junk",
              "ungrouped single outliers"],
    "unique_notes": [int((sing["note_clean"].str.split().str.len() > 50).sum()),
                     int(sing["note_clean"].str.contains(r"https?://", case=False).sum()),
                     int(sing["note_clean"].str.fullmatch(r"(?i)(test( only)?|[a-z]{1,8})").sum()),
                     len(sing)]})
print(f"Outliers: {len(out_idx)} notes / {w[out_idx].sum()} tickets -> {len(outlier_cards)} mini-themes + {len(singles)} ungrouped notes")
display(dq)
# ungrouped singles go to the LLM as one sampled card so that unusual reasons still get looked at
rng = np.random.default_rng(CONFIG["random_seed"])
single_sample = rng.choice(singles, size=min(60, len(singles)), replace=False) if singles else []
outlier_singles_card = {"cluster_id": "O-SINGLES", "kind": "outlier_singles", "unique_notes": len(singles),
                        "records": int(w[singles].sum()) if singles else 0, "keywords": [],
                        "sample_notes": [fmt(i) for i in single_sample]}

In [ ]:
summary = pd.DataFrame([{k: c[k] for k in ("cluster_id", "kind", "unique_notes", "records", "mean_within_similarity", "mean_membership_probability")}
                        | {"keywords": ", ".join(c["keywords"][:6]), "top_note": c["representative_notes"][0][:90]}
                        for c in cards + outlier_cards])
display(summary)
(ART / "05_cluster_cards.json").write_text(json.dumps({"clusters": cards, "outlier_groups": outlier_cards,
    "outlier_singles": outlier_singles_card}, indent=1, ensure_ascii=False), encoding="utf-8")
unique_notes.to_pickle(ART / "05_unique_notes.pkl")
RUN["status"]["phase5"] = "done"

## Phase 6 · LLM taxonomy discovery → consolidation → human review

**6a.** Every cluster card (plus outlier mini-themes and a sample of ungrouped outliers) goes to the LLM with the discovery prompt. The LLM returns one JSON object per cluster, which is validated against `DISCOVERY_SCHEMA`.
**6b.** A second pass consolidates the cluster verdicts into one proposed taxonomy, with definitions, inclusion/exclusion rules, examples and subcategories.
**6c.** The proposed taxonomy is exported to `taxonomy/taxonomy_review_v1.xlsx` for a human to edit. Classification (Phase 7) reads **only** the approved JSON built from that file, never the LLM proposal directly.

In [ ]:
BASELINE = {
    "RESIGNATION": "An individual has left their role or organization, resigned, retired, or is no longer responsible for the service or account.",
    "SERVICE_DISSATISFACTION": "The customer is unhappy with the service, product, support, quality, performance, experience, or perceived value.",
    "BUSINESS_REASON": "A business decision, organizational change, financial circumstance, operational requirement, or other commercial consideration has led to disengagement.",
    "FOR_RECORD_PURPOSE_ONLY": "The ticket exists for administrative, documentation, tracking, or procedural purposes rather than to communicate an actual reason for disengagement.",
    "DELETED": "The note explicitly indicates that a ticket, account, service, or record was deleted or removed, without providing a clearer underlying reason.",
    "BLANK": "The note is empty, null, whitespace-only, or contains no meaningful information.",
}

TAXONOMY_DISCOVERY_PROMPT = '''# Role
You are a customer churn analytics and taxonomy discovery specialist.

# Context
Cloudstaff is an offshore staffing provider. Each ticket is a "disengagement": one Cloudstaffer's engagement with a client ends
(staff resigns, client releases the staff, client leaves Cloudstaff, or an administrative re-papering). The note is free text typed by
account managers, sometimes a pasted client email, sometimes a form value ("Voluntary", "Involuntary", "For Record Purpose Only").

# Objective
Identify, group and categorise the underlying reasons for disengagement. Use the baseline categories as a starting point, but do NOT
assume they cover every reason; propose new categories where the data supports them.

# Baseline categories
{baseline}

# Rules
1. Test each cluster against the baseline first. 2. Do not force a cluster into a baseline category it does not reasonably fit.
3. Identify recurring reasons outside the baseline; propose new categories only when several notes show a distinct, recurring reason.
4. Do not create a category per wording variant or isolated example; consolidate synonyms.
5. Distinguish the immediate reason from the underlying business reason when the notes support it.
6. Preserve distinctions such as resignation, service dissatisfaction, pricing, budget constraints, business closure, competitor
   displacement, low adoption and restructuring - only where the notes show them.
7. Do not assume dissatisfaction because someone stopped responding or stopped using the service. Do not infer unsupported information.
8. Notes with several reasons: identify the dominant reason and contributing reasons.
9. A cluster with materially different reasons is MIXED and should be SPLIT.
10. Use UNKNOWN when a meaningful reason exists but cannot be mapped confidently. Keep BLANK and FOR_RECORD_PURPOSE_ONLY distinct.
11. Separate administrative ticket activity from real disengagement reasons.

# Input
Per cluster: id, unique-note count, record (ticket) count, keywords, representative / closest / furthest notes. "[n]" before a note =
number of tickets with that exact note. Cluster ids starting with "O" are outlier mini-themes; "O-SINGLES" is a sample of ungrouped outliers.

# Output
A JSON array, one object per input cluster, each matching:
{{"cluster_id": str, "cluster_size": int, "dominant_theme": str,
 "baseline_category_match": "RESIGNATION|SERVICE_DISSATISFACTION|BUSINESS_REASON|FOR_RECORD_PURPOSE_ONLY|DELETED|BLANK|NONE",
 "proposed_category": str, "category_type": "EXISTING|NEW|UNKNOWN", "category_definition": str,
 "proposed_subcategories": [str], "supporting_examples": [str], "contradictory_examples": [str],
 "cluster_type": "PURE|MIXED|UNCERTAIN", "confidence": 0-1, "recommended_action": "KEEP|MERGE|SPLIT|REVIEW"}}
'''.format(baseline="\n".join(f"- {k}: {v}" for k, v in BASELINE.items()))

DISCOVERY_ITEM = {
    "type": "object",
    "required": ["cluster_id", "cluster_size", "dominant_theme", "baseline_category_match", "proposed_category", "category_type",
                 "category_definition", "proposed_subcategories", "supporting_examples", "contradictory_examples",
                 "cluster_type", "confidence", "recommended_action"],
    "properties": {
        "cluster_id": {"type": "string"}, "cluster_size": {"type": "integer", "minimum": 0},
        "dominant_theme": {"type": "string", "minLength": 3},
        "baseline_category_match": {"enum": list(BASELINE) + ["NONE"]},
        "proposed_category": {"type": "string", "pattern": "^[A-Z][A-Z_]+$"},
        "category_type": {"enum": ["EXISTING", "NEW", "UNKNOWN"]},
        "category_definition": {"type": "string", "minLength": 10},
        "proposed_subcategories": {"type": "array", "items": {"type": "string"}},
        "supporting_examples": {"type": "array", "items": {"type": "string"}, "minItems": 1},
        "contradictory_examples": {"type": "array", "items": {"type": "string"}},
        "cluster_type": {"enum": ["PURE", "MIXED", "UNCERTAIN"]},
        "confidence": {"type": "number", "minimum": 0, "maximum": 1},
        "recommended_action": {"enum": ["KEEP", "MERGE", "SPLIT", "REVIEW"]},
        "split_into": {"type": "array", "items": {"type": "string"}},
    }}
DISCOVERY_SCHEMA = {"type": "array", "items": DISCOVERY_ITEM, "minItems": 1}

all_cards = cards + outlier_cards + [outlier_singles_card]
discovery_payload = [{k: v for k, v in c.items() if k != "kind"} | {"kind": c["kind"]} for c in all_cards]
try:
    discovery = llm_exchange("06a_taxonomy_discovery", TAXONOMY_DISCOVERY_PROMPT, discovery_payload, DISCOVERY_SCHEMA)
except LLMResponsePending as e:
    discovery = None; print(e)

In [ ]:
if discovery is not None:
    disc = pd.DataFrame(discovery)
    sizes = {c["cluster_id"]: c["records"] for c in all_cards}
    missing = set(sizes) - set(disc["cluster_id"]); extra = set(disc["cluster_id"]) - set(sizes)
    bad_size = disc[disc["cluster_size"] != disc["cluster_id"].map(sizes)]
    if missing or extra or len(bad_size):
        raise ValueError(f"Discovery response incomplete: missing={missing} extra={extra} size_mismatch={bad_size['cluster_id'].tolist()}")
    print(f"{len(disc)} cluster verdicts validated")
    display(disc.groupby(["proposed_category"]).agg(clusters=("cluster_id", "size"), records=("cluster_size", "sum"))
            .sort_values("records", ascending=False))
    display(pd.crosstab(disc["cluster_type"], disc["recommended_action"]))
    print("Clusters flagged MIXED / SPLIT:")
    display(disc.loc[disc["cluster_type"].ne("PURE") | disc["recommended_action"].eq("SPLIT"),
                     ["cluster_id", "cluster_size", "dominant_theme", "proposed_category", "cluster_type", "recommended_action", "split_into"]])

In [ ]:
TAXONOMY_CONSOLIDATION_PROMPT = '''You receive the cluster-level taxonomy verdicts produced earlier (JSON array) for Cloudstaff disengagement notes.
Consolidate them into ONE practical taxonomy:
- merge clusters/categories that are the same underlying reason; split categories that are too broad;
- keep categories separate when they reflect different business realities (who initiated, controllable vs market, admin vs churn);
- use parent categories with subcategories rather than many narrow labels;
- always include the six baseline categories (they may be empty or redefined only with justification), plus UNKNOWN;
- give each category: id, name, definition, inclusion rules, exclusion rules, example notes, subcategories, origin (BASELINE|NEW|SYSTEM),
  supporting cluster ids and an estimated ticket count;
- list open questions a human reviewer should decide.
Return JSON: {"categories": [...], "open_questions": [str]}'''

CATEGORY_ITEM = {
    "type": "object",
    "required": ["category_id", "category_name", "definition", "inclusion_rules", "exclusion_rules", "example_notes",
                 "subcategories", "origin", "supporting_cluster_ids", "estimated_records"],
    "properties": {
        "category_id": {"type": "string", "pattern": "^[A-Z][A-Z_]+$"},
        "category_name": {"type": "string", "minLength": 3},
        "definition": {"type": "string", "minLength": 15},
        "inclusion_rules": {"type": "array", "items": {"type": "string"}, "minItems": 1},
        "exclusion_rules": {"type": "array", "items": {"type": "string"}},
        "example_notes": {"type": "array", "items": {"type": "string"}},
        "subcategories": {"type": "array", "items": {"type": "string"}},
        "origin": {"enum": ["BASELINE", "NEW", "SYSTEM"]},
        "supporting_cluster_ids": {"type": "array", "items": {"type": "string"}},
        "estimated_records": {"type": "integer", "minimum": 0},
        "reviewer_notes": {"type": "string"},
    }}
CONSOLIDATION_SCHEMA = {"type": "object", "required": ["categories", "open_questions"],
                        "properties": {"categories": {"type": "array", "items": CATEGORY_ITEM, "minItems": 7},
                                       "open_questions": {"type": "array", "items": {"type": "string"}}}}
proposed = None
if discovery is not None:
    try:
        proposed = llm_exchange("06b_taxonomy_consolidation", TAXONOMY_CONSOLIDATION_PROMPT, discovery, CONSOLIDATION_SCHEMA)
    except LLMResponsePending as e:
        print(e)
if proposed is not None:
    ids = [c["category_id"] for c in proposed["categories"]]
    assert len(ids) == len(set(ids)), "duplicate category ids"
    assert set(BASELINE) | {"UNKNOWN"} <= set(ids), "baseline categories and UNKNOWN must be present"
    unmapped = set(disc["cluster_id"]) - {c for cat in proposed["categories"] for c in cat["supporting_cluster_ids"]}
    if unmapped: warn(f"Clusters not referenced by any proposed category: {sorted(unmapped)}")
    prop = pd.DataFrame(proposed["categories"])
    display(prop[["category_id", "category_name", "origin", "estimated_records", "subcategories"]]
            .assign(subcategories=lambda x: x["subcategories"].str.join("; ")))
    print("Open questions for the reviewer:")
    for q in proposed["open_questions"]: print(" -", q)

In [ ]:
# ---- 6c. Export for human review -------------------------------------------
# The review workbook is the human's working copy and can be reopened at any time (see "Revisiting the taxonomy").
# It is only rewritten when missing or on an older layout; then every reviewer column is carried over and the old file backed up.
REVIEW_XLSX = TAX / "taxonomy_review_v1.xlsx"
REVIEW_LAYOUT = 3           # 2 = redirect_to on Cluster mapping; 3 = revisit / revisit_note columns on both sheets
REVIEW_HELP = [
    ("Taxonomy: approve", "Y = keep this category, N = remove it. Every row needs Y or N before approval."),
    ("Taxonomy: category_id / category_name", "Rename freely. Use UPPER_SNAKE_CASE for the id. Redirects that point at the old id must be updated."),
    ("Taxonomy: merge_into", "To merge, put the target category_id here and set approve = N. Rules, examples and clusters carry over."),
    ("Taxonomy: split / add", "Add new rows (approve = Y, definition, at least one inclusion rule). Set approve = N on a row you split up."),
    ("Taxonomy: definition / rules / examples / subcategories", "Edit the text. Lists are separated by ' | '."),
    ("Taxonomy: reviewer_notes", "Anything the classifier should know. It's passed into the classification prompt."),
    ("Cluster mapping: redirect_to", "Pick a category_id from the dropdown to send this cluster to a different category than the LLM proposed. "
                                     "Leave blank to accept proposed_category. Must be an approved category, or one merged into an approved category."),
    ("Cluster mapping: reviewer_notes", "Why you redirected, or a note on the cluster."),
    ("Cluster mapping: what a redirect changes", "Clusters are a discovery aid: a redirect moves the cluster's example notes and supporting-cluster link to the new "
                                                 "category. Each ticket is still classified on its own note."),
    ("Both sheets: revisit / revisit_note", "Set revisit = Y on anything you approved provisionally and want to come back to. It does NOT block approval; "
                                            "flagged items are listed in the output workbook ('Revisit list') until you clear the flag."),
    ("Revisiting later", "Edit this file at any time and re-run the notebook. A changed taxonomy gets a new version, is saved to taxonomy/history/, "
                         "the change log shows what moved, and only the classifications for the new version are recomputed. Earlier versions stay cached."),
]
LIST_COLS = ["inclusion_rules", "exclusion_rules", "example_notes", "subcategories", "supporting_cluster_ids"]
TAX_EDIT_COLS = ["approve", "merge_into", "reviewer_notes", "revisit", "revisit_note"]
MAP_EDIT_COLS = ["redirect_to", "reviewer_notes", "revisit", "revisit_note"]
MAP_COLS = ["cluster_id", "cluster_size", "dominant_theme", "top_notes", "proposed_category", "redirect_to", "reviewer_notes",
            "revisit", "revisit_note", "cluster_type", "recommended_action", "split_into", "confidence"]

def proposed_taxonomy_frame():
    out = prop.copy()
    for c in LIST_COLS: out[c] = out[c].map(lambda v: " | ".join(v))
    out.insert(0, "approve", ""); out.insert(1, "merge_into", "")
    for c in ("reviewer_notes", "revisit", "revisit_note"):
        if c not in out: out[c] = ""
    return out

def proposed_mapping_frame():
    top = {c["cluster_id"]: "\n".join(n[:160] for n in (c.get("representative_notes") or c.get("sample_notes", []))[:3]) for c in all_cards}
    m = disc.copy()
    m["top_notes"] = m["cluster_id"].map(top)
    m["split_into"] = m.get("split_into", pd.Series([None] * len(m))).map(lambda v: ", ".join(v) if isinstance(v, list) else "")
    for c in MAP_EDIT_COLS: m[c] = ""
    return m[MAP_COLS]

def write_review_workbook(tax, cmap, path):
    with pd.ExcelWriter(path, engine="xlsxwriter") as xw:
        tax.to_excel(xw, sheet_name="Taxonomy", index=False)
        cmap.to_excel(xw, sheet_name="Cluster mapping", index=False)
        pd.DataFrame(REVIEW_HELP, columns=["column / action", "how"]).to_excel(xw, sheet_name="How to review", index=False)
        pd.DataFrame({"open_question": proposed["open_questions"]}).to_excel(xw, sheet_name="Open questions", index=False)
        pd.DataFrame({"layout": [REVIEW_LAYOUT]}).to_excel(xw, sheet_name="_meta", index=False)
        wb = xw.book; wrap = wb.add_format({"text_wrap": True, "valign": "top"})
        edit = wb.add_format({"text_wrap": True, "valign": "top", "bg_color": "#FFF7D6"})   # columns the reviewer fills
        id_col = tax.columns.get_loc("category_id")
        id_range = f"=Taxonomy!${chr(65 + id_col)}$2:${chr(65 + id_col)}$200"     # live list: new/renamed categories appear in dropdowns
        ws = xw.sheets["Taxonomy"]; ws.freeze_panes(1, 3); ws.autofilter(0, 0, len(tax), len(tax.columns) - 1)
        widths = {"approve": 9, "merge_into": 14, "category_id": 28, "category_name": 30, "definition": 60, "origin": 10,
                  "estimated_records": 10, "revisit": 8, "revisit_note": 35}
        for i, col in enumerate(tax.columns):
            ws.set_column(i, i, widths.get(col, 45), edit if col in TAX_EDIT_COLS else wrap)
        ws.data_validation(1, tax.columns.get_loc("approve"), 199, tax.columns.get_loc("approve"), {"validate": "list", "source": ["Y", "N"]})
        ws.data_validation(1, tax.columns.get_loc("merge_into"), 199, tax.columns.get_loc("merge_into"), {"validate": "list", "source": id_range, "ignore_blank": True})
        ws.data_validation(1, tax.columns.get_loc("revisit"), 199, tax.columns.get_loc("revisit"), {"validate": "list", "source": ["Y"], "ignore_blank": True})
        ws = xw.sheets["Cluster mapping"]; ws.freeze_panes(1, 1); ws.autofilter(0, 0, len(cmap), len(cmap.columns) - 1)
        mw = {"cluster_id": 10, "cluster_size": 9, "dominant_theme": 45, "top_notes": 70, "proposed_category": 30, "redirect_to": 30,
              "reviewer_notes": 35, "revisit": 8, "revisit_note": 35, "cluster_type": 11, "recommended_action": 11, "split_into": 40, "confidence": 9}
        for i, col in enumerate(cmap.columns):
            ws.set_column(i, i, mw.get(col, 20), edit if col in MAP_EDIT_COLS else wrap)
        rc = cmap.columns.get_loc("redirect_to")
        ws.data_validation(1, rc, len(cmap), rc, {"validate": "list", "source": id_range, "ignore_blank": True,
                                                  "input_title": "Redirect cluster", "input_message": "Pick a category_id, or leave blank to keep proposed_category"})
        vc = cmap.columns.get_loc("revisit")
        ws.data_validation(1, vc, len(cmap), vc, {"validate": "list", "source": ["Y"], "ignore_blank": True})
        xw.sheets["How to review"].set_column(0, 0, 42); xw.sheets["How to review"].set_column(1, 1, 110, wrap)
        xw.sheets["Open questions"].set_column(0, 0, 140, wrap)
        xw.sheets["_meta"].hide()

def review_layout(path):
    try:
        return int(pd.read_excel(path, "_meta")["layout"].iloc[0])
    except Exception:
        return 1

if proposed is not None:
    if not REVIEW_XLSX.exists():
        write_review_workbook(proposed_taxonomy_frame(), proposed_mapping_frame(), REVIEW_XLSX)
        print(f"Proposed taxonomy written for review: {REVIEW_XLSX}")
    elif review_layout(REVIEW_XLSX) < REVIEW_LAYOUT:
        backup = TAX / f"taxonomy_review_v1.backup-{dt.datetime.now():%Y%m%d-%H%M%S}.xlsx"
        backup.write_bytes(REVIEW_XLSX.read_bytes())
        kept_tax = pd.read_excel(REVIEW_XLSX, "Taxonomy", dtype=object)          # reviewer's sheet, kept exactly
        for c in ("revisit", "revisit_note"):
            if c not in kept_tax: kept_tax[c] = ""
        old_map = pd.read_excel(REVIEW_XLSX, "Cluster mapping", dtype=object)
        new_map = proposed_mapping_frame()
        for col in MAP_EDIT_COLS:                                                  # carry over any mapping edits
            if col in old_map:
                new_map[col] = new_map["cluster_id"].map(old_map.set_index("cluster_id")[col]).fillna("")
        write_review_workbook(kept_tax, new_map, REVIEW_XLSX)
        n_app = kept_tax["approve"].astype(str).str.strip().str.upper().isin(["Y", "N"]).sum()
        print(f"Review workbook upgraded to layout {REVIEW_LAYOUT}; your sheets were kept ({n_app} rows approved). Backup: {backup.name}")
    else:
        print(f"{REVIEW_XLSX.name} is up to date - not overwritten (it holds your edits).")

### Revisiting the taxonomy

Approval isn't final. To revisit:
1. Open `taxonomy/taxonomy_review_v1.xlsx`. Change approvals, merges, definitions, rules or redirects, or set **`revisit = Y`** (plus a `revisit_note`) on anything you approved provisionally. A revisit flag doesn't block approval.
2. Re-run the notebook. 6d builds a new approved version if anything changed: the version id is a hash of the content. It also saves a copy to `taxonomy/history/` and prints a change log against the previous approved version.
3. Phase 7 classifies against the current version only. Results for each version are cached separately, so going back to an earlier version doesn't redo work, and stale results are never mixed in.

Open revisit flags are listed on the output workbook's **Revisit list** sheet, and the **Taxonomy history** sheet lists every approved version.

In [ ]:
# ---- 6d. Build the APPROVED taxonomy from the reviewed file ------------------
def _flag(v):
    return str(v).strip().upper() in ("Y", "YES", "TRUE", "1")

def build_approved_taxonomy(path):
    t = pd.read_excel(path, "Taxonomy", dtype=object).fillna("")
    for c in ("revisit", "revisit_note"):
        if c not in t: t[c] = ""
    t["approve"] = t["approve"].astype(str).str.strip().str.upper()
    t["category_id"] = t["category_id"].astype(str).str.strip()
    t["merge_into"] = t["merge_into"].astype(str).str.strip()
    pending = t.loc[~t["approve"].isin(["Y", "N"]), "category_id"].tolist()
    if pending:
        return None, f"{len(pending)} rows still need approve = Y/N: {pending}"
    keep = t[t["approve"] == "Y"].copy()
    approved_ids = set(keep["category_id"])
    final = {r["category_id"]: (r["category_id"] if r["approve"] == "Y" else (r["merge_into"] or None)) for _, r in t.iterrows()}
    bad_merge = {k: v for k, v in final.items() if v and v not in approved_ids}
    if bad_merge:
        return None, f"merge_into must point at an approved category: {bad_merge}"
    for _, r in t[(t["approve"] == "N") & (t["merge_into"] != "")].iterrows():
        tgt = keep["category_id"] == r["merge_into"]
        for c in ["inclusion_rules", "example_notes", "subcategories"]:
            if str(r[c]).strip():
                keep.loc[tgt, c] = keep.loc[tgt, c].astype(str) + " | " + str(r[c])

    m = pd.read_excel(path, "Cluster mapping", dtype=object).fillna("")
    for c in MAP_EDIT_COLS:
        if c not in m: m[c] = ""
    m["redirect_to"] = m["redirect_to"].astype(str).str.strip()
    m["chosen"] = np.where(m["redirect_to"] != "", m["redirect_to"], m["proposed_category"])
    unknown_target = sorted(set(m.loc[m["redirect_to"] != "", "redirect_to"]) - set(final))
    if unknown_target:
        return None, f"redirect_to names categories that are not in the Taxonomy sheet: {unknown_target}"
    m["approved_category"] = m["chosen"].map(lambda c: final.get(c))
    orphan = m[m["approved_category"].isna()]
    if len(orphan):
        return None, ("These clusters point at a removed category - set redirect_to for them: "
                      + ", ".join(f"{r.cluster_id} ({r.chosen})" for r in orphan.itertuples()))
    cluster_mapping = [{"cluster_id": r.cluster_id, "cluster_size": int(r.cluster_size), "proposed_category": r.proposed_category,
                        "redirect_to": r.redirect_to or None, "approved_category": r.approved_category,
                        "reviewer_notes": str(r.reviewer_notes), "revisit": _flag(r.revisit), "revisit_note": str(r.revisit_note)}
                       for r in m.itertuples()]

    cards_by_id = {c["cluster_id"]: c for c in all_cards}
    moved = m[m["redirect_to"] != ""]
    cats = []
    for _, r in keep.iterrows():
        cat = {k: str(r[k]) for k in ["category_id", "category_name", "definition", "origin", "reviewer_notes"]}
        for c in LIST_COLS: cat[c] = [x.strip() for x in str(r[c]).split("|") if x.strip()]
        cid = cat["category_id"]
        cat["supporting_cluster_ids"] = sorted(m.loc[m["approved_category"] == cid, "cluster_id"])
        for mr in moved[moved["approved_category"] == cid].itertuples():
            notes = cards_by_id.get(mr.cluster_id, {}).get("representative_notes", [])[:2]
            cat["example_notes"] += [re.sub(r"^\[\d+\]\s*", "", n)[:200] for n in notes]
        cat["estimated_records"] = int(m.loc[m["approved_category"] == cid, "cluster_size"].sum())   # tickets in clusters mapped here
        cat["revisit"] = _flag(r["revisit"]); cat["revisit_note"] = str(r["revisit_note"])
        cats.append(cat)
    ids = [c["category_id"] for c in cats]
    if len(ids) != len(set(ids)): return None, "duplicate category_id after review"
    for req in ("BLANK", "UNKNOWN"):
        if req not in ids: return None, f"{req} must stay in the approved taxonomy"
    jsonschema.validate({"categories": cats, "open_questions": []}, CONSOLIDATION_SCHEMA)
    # the version covers what changes classification (definitions, rules, examples, mapping), not revisit flags or notes-to-self
    content = {"categories": [{k: v for k, v in c.items() if k not in ("revisit", "revisit_note", "estimated_records")} for c in cats],
               "clusters": [{k: v for k, v in c.items() if k not in ("revisit", "revisit_note", "reviewer_notes")} for c in cluster_mapping]}
    version = f"v1-{sha(json.dumps(content, sort_keys=True, ensure_ascii=False), 8)}"
    return {"taxonomy_version": version, "approved_at": dt.datetime.now().isoformat(timespec="seconds"),
            "source_file": path.name, "categories": cats, "cluster_mapping": cluster_mapping}, None

def taxonomy_changelog(old, new):
    if old is None: return pd.DataFrame([{"change": "first approved version", "item": new["taxonomy_version"], "detail": ""}])
    rows = []
    oc = {c["category_id"]: c for c in old["categories"]}; nc = {c["category_id"]: c for c in new["categories"]}
    rows += [{"change": "category added", "item": k, "detail": nc[k]["category_name"]} for k in nc.keys() - oc.keys()]
    rows += [{"change": "category removed", "item": k, "detail": oc[k]["category_name"]} for k in oc.keys() - nc.keys()]
    for k in nc.keys() & oc.keys():
        for f in ("category_name", "definition", "inclusion_rules", "exclusion_rules", "subcategories", "reviewer_notes"):
            if nc[k].get(f) != oc[k].get(f): rows.append({"change": f"{f} edited", "item": k, "detail": str(nc[k].get(f))[:300]})
        if nc[k].get("revisit") != oc[k].get("revisit"):
            rows.append({"change": "revisit flag " + ("set" if nc[k].get("revisit") else "cleared"), "item": k, "detail": nc[k].get("revisit_note", "")})
    om = {c["cluster_id"]: c for c in old.get("cluster_mapping", [])}
    for c in new.get("cluster_mapping", []):
        o = om.get(c["cluster_id"])
        if o and o["approved_category"] != c["approved_category"]:
            rows.append({"change": "cluster remapped", "item": c["cluster_id"], "detail": f"{o['approved_category']} -> {c['approved_category']}"})
    return pd.DataFrame(rows or [{"change": "no change", "item": new["taxonomy_version"], "detail": ""}])

APPROVED_JSON = TAX / "taxonomy_approved_v1.json"      # always the CURRENT approved version
HIST = TAX / "history"; HIST.mkdir(exist_ok=True)
approved, changelog = None, None
if REVIEW_XLSX.exists():
    previous = json.loads(APPROVED_JSON.read_text(encoding="utf-8")) if APPROVED_JSON.exists() else None
    approved, problem = build_approved_taxonomy(REVIEW_XLSX)
    if problem:
        print("Taxonomy NOT approved yet:", problem)
        if previous:
            print(f"Keeping the previous approved version {previous['taxonomy_version']} until the review file is complete.")
            approved = previous
    else:
        if previous and previous["taxonomy_version"] == approved["taxonomy_version"]:
            approved["approved_at"] = previous["approved_at"]              # same content: keep the original approval time
        changelog = taxonomy_changelog(previous, approved)
        APPROVED_JSON.write_text(json.dumps(approved, indent=1, ensure_ascii=False), encoding="utf-8")
        hist_file = HIST / f"taxonomy_approved_{approved['taxonomy_version']}.json"
        if not hist_file.exists():
            hist_file.write_text(json.dumps(approved, indent=1, ensure_ascii=False), encoding="utf-8")
        red = sum(1 for c in approved["cluster_mapping"] if c["redirect_to"])
        flags = sum(c["revisit"] for c in approved["categories"]) + sum(c["revisit"] for c in approved["cluster_mapping"])
        print(f"Approved taxonomy {approved['taxonomy_version']}: {len(approved['categories'])} categories, "
              f"{red} clusters redirected, {flags} items flagged to revisit")
        print("Change log vs previous approved version:"); display(changelog)
RUN["status"]["phase6"] = f"approved {approved['taxonomy_version']}" if approved else "awaiting human review"

## Phase 7 · Classify every ticket against the approved taxonomy

Each **unique cleaned note** is classified individually by the LLM, and the result is copied to every ticket with that exact note. Identical text always gets an identical answer, so classifying it once per ticket would only add cost.
* **Batches** are grouped by size (`classification_batch_chars`). Each batch is named by a hash of its note keys, so a rerun after a failure only creates new batches for the notes that failed.
* **Cache:** `llm/classification/cache.jsonl`. The key is a hash of (cleaned note, taxonomy version, model, prompt version), so changing the taxonomy or the prompt never reuses stale answers.
* **Validation:** every item is checked against `CLS_ITEM_SCHEMA` and the approved category ids. Invalid or missing items are re-requested and never silently accepted.
* **Review rules** are enforced in code as well as asked of the LLM: confidence below `confidence_threshold`, UNKNOWN type with a real category, and others.
* **BLANK** notes (Phase 2) don't go to the LLM; they get `BLANK` by rule.
* **Levels:** `main_category` comes from the category (`main_category_map`). `detail_category` comes from the category and subcategory (`detail_category_map`; `<category>_general` when there's no subcategory). STAFF_MEDICAL merges into `resignation_health`.
* `classification_source` records where a label came from:
  * `llm` (the note);
  * `form fields` (step 7b);
  * `rule: blank note`;
  * `rule: test ticket` (→ BLANK);
  * `rule: cancelled ticket` (→ DELETED).

  The two newer rules run at the end of 7b.

In [ ]:
CONFIG.setdefault("classification_batch_chars", 40000)
if approved is None:
    raise RuntimeError("No approved taxonomy - finish Phase 6 review first.")
TAXV = approved["taxonomy_version"]; PV = CONFIG["classification_prompt_version"]; MODEL = CONFIG["llm_model"]
CAT = {c["category_id"]: c for c in approved["categories"]}
CAT_IDS = sorted(CAT)

CLASSIFICATION_PROMPT = '''# Task
Classify Cloudstaff disengagement notes against the APPROVED taxonomy below. Each note explains why one Cloudstaffer's engagement
with a client ended (or is an administrative record). Classify each note on its own text only.

# Rules
- primary_reason: the ONE category that best states the main reason. Use exactly one approved category_id.
- primary_subcategory: one of that category's listed subcategories if the note supports it, else null.
- secondary_reasons: other approved category_ids ONLY if the note explicitly states an additional contributing reason. Never repeat
  the primary. Do not add semantically related categories that are not stated.
- classification_type: EXPLICIT (note directly states the reason), INFERRED (reasonably implied, not stated), UNKNOWN (text exists but
  no reliable reason -> primary_reason must be UNKNOWN, or BLANK if the text is meaningless).
- BLANK = empty / meaningless text (e.g. 'test', random characters). UNKNOWN = meaningful text with no assignable reason. Keep distinct.
- Never infer a reason from the mere fact that the staff or client disengaged. An unknown reason is better than an invented one.
- Follow each category's inclusion and exclusion rules and the reviewer notes.
- evidence: the shortest exact snippet of the note that supports the primary reason ('' if none).
- confidence: 0-1. review_required = true when confidence < {thr}, several primary categories are plausible, the note is ambiguous,
  has several unrelated reasons, does not fit the taxonomy, or EXPLICIT vs INFERRED is unclear. review_reason: short text, else null.

# Approved taxonomy ({taxv})
{taxonomy}

# Output
A JSON array with one object per input note, in any order:
{{"note_key": str, "primary_reason": category_id, "primary_subcategory": str|null, "secondary_reasons": [category_id],
  "classification_type": "EXPLICIT|INFERRED|UNKNOWN", "evidence": str, "confidence": number, "review_required": bool, "review_reason": str|null}}
'''.format(thr=CONFIG["confidence_threshold"], taxv=TAXV, taxonomy=json.dumps(
    [{k: c[k] for k in ("category_id", "category_name", "definition", "inclusion_rules", "exclusion_rules", "subcategories", "reviewer_notes")}
     | {"example_notes": c["example_notes"][:6]} for c in approved["categories"]], indent=1, ensure_ascii=False))

CLS_ITEM_SCHEMA = {
    "type": "object",
    "required": ["note_key", "primary_reason", "primary_subcategory", "secondary_reasons", "classification_type", "evidence",
                 "confidence", "review_required", "review_reason"],
    "properties": {
        "note_key": {"type": "string"}, "primary_reason": {"enum": CAT_IDS},
        "primary_subcategory": {"type": ["string", "null"]},
        "secondary_reasons": {"type": "array", "items": {"enum": CAT_IDS}, "uniqueItems": True},
        "classification_type": {"enum": ["EXPLICIT", "INFERRED", "UNKNOWN"]},
        "evidence": {"type": "string"}, "confidence": {"type": "number", "minimum": 0, "maximum": 1},
        "review_required": {"type": "boolean"}, "review_reason": {"type": ["string", "null"]},
    }}
print(f"Taxonomy {TAXV}: {len(CAT_IDS)} categories; prompt {PV}; prompt length {len(CLASSIFICATION_PROMPT):,} chars")

In [ ]:
CLS_DIR = LLM_DIR / "classification"; CACHE_FILE = CLS_DIR / "cache.jsonl"
def cache_key(note_clean):
    return sha(f"{note_clean}\x1f{TAXV}\x1f{MODEL}\x1f{PV}", 24)

def load_cache():
    out = {}
    if CACHE_FILE.exists():
        for line in CACHE_FILE.read_text(encoding="utf-8").splitlines():
            if line.strip():
                rec = json.loads(line); out[rec["key"]] = rec
    return out

def check_item(item, valid_keys):
    # returns (cleaned item, problem) - problem is None when the item is usable
    try:
        jsonschema.validate(item, CLS_ITEM_SCHEMA)
    except jsonschema.ValidationError as e:
        return None, f"schema: {e.message[:120]}"
    if item["note_key"] not in valid_keys:
        return None, "unknown note_key"
    it = dict(item)
    it["secondary_reasons"] = [s for s in it["secondary_reasons"] if s != it["primary_reason"]]
    subs = CAT[it["primary_reason"]]["subcategories"]
    if it["primary_subcategory"] is not None and it["primary_subcategory"] not in subs:
        it["primary_subcategory"] = None
    return it, None

cache = load_cache()
todo = unique_notes[[cache_key(n) not in cache for n in unique_notes["note_clean"]]].copy()
batches, cur, size = [], [], 0
for r in todo.sort_values("note_key").itertuples():                       # deterministic grouping
    L = len(r.note_clean)
    if cur and size + L > CONFIG["classification_batch_chars"]:
        batches.append(cur); cur, size = [], 0
    cur.append(r); size += L
if cur: batches.append(cur)

pending, got, failed = [], 0, []
for b in batches:
    keys = [r.note_key for r in b]
    bid = f"{TAXV}_{PV}_" + sha("|".join(keys), 10)
    payload = [{"note_key": r.note_key, "note_original": str(r.note_original), "note_clean": r.note_clean, "tickets": int(r.records)} for r in b]
    try:
        resp = llm_exchange(bid, CLASSIFICATION_PROMPT, payload, schema={"type": "array"}, subdir="classification")
    except LLMResponsePending:
        pending.append(bid); continue
    by_key = {r.note_key: r for r in b}; seen = set()
    with CACHE_FILE.open("a", encoding="utf-8") as fh:
        for item in resp:
            it, problem = check_item(item, by_key.keys())
            if problem or it["note_key"] in seen:
                failed.append((bid, item.get("note_key"), problem or "duplicate")); continue
            seen.add(it["note_key"])
            rec = {"key": cache_key(by_key[it["note_key"]].note_clean), "note_key": it["note_key"], "taxonomy_version": TAXV,
                   "prompt_version": PV, "model": MODEL, "batch": bid, "result": it}
            fh.write(json.dumps(rec, ensure_ascii=False) + "\n"); got += 1
    for k in by_key.keys() - seen:
        failed.append((bid, k, "missing from response"))
logger.info(f"Classification: {len(batches)} batches, {got} new results, {len(failed)} failed, {len(pending)} pending")
print(f"{len(unique_notes):,} unique notes | cached before this run: {len(unique_notes) - len(todo):,} | "
      f"batches: {len(batches)} | new results: {got} | failed items: {len(failed)} | batches awaiting a response: {len(pending)}")
if failed:
    RUN["errors"].append(f"{len(failed)} classification items failed validation and will be re-requested on the next run")
    display(pd.DataFrame(failed, columns=["batch", "note_key", "problem"]).head(20))
if pending:
    print("Awaiting LLM responses for:", *pending, sep="\n  ")

In [ ]:
cache = load_cache()
res = {}
for n, k in zip(unique_notes["note_clean"], unique_notes["note_key"]):
    rec = cache.get(cache_key(n))
    if rec: res[k] = rec["result"]
complete = len(res) == len(unique_notes)
print(f"Classified unique notes: {len(res):,} / {len(unique_notes):,}" + ("" if complete else "  (incomplete - rerun after the pending batches are answered)"))

def apply_review_rules(it):
    reasons = [it["review_reason"]] if it.get("review_reason") else []
    flag = bool(it["review_required"])
    if it["confidence"] < CONFIG["confidence_threshold"]:
        flag = True; reasons.append(f"confidence below {CONFIG['confidence_threshold']}")
    if it["classification_type"] == "UNKNOWN" and it["primary_reason"] not in ("UNKNOWN", "BLANK"):
        flag = True; reasons.append("type UNKNOWN but a reason category was assigned")
    if it["primary_reason"] == "UNKNOWN" and it["classification_type"] != "UNKNOWN":
        flag = True; reasons.append("UNKNOWN category with a non-UNKNOWN type")
    return flag, "; ".join(dict.fromkeys(reasons)) or None

OUT_COLS = ["primary_reason_id", "primary_reason_name", "primary_subcategory", "secondary_reason_ids", "secondary_reason_names",
            "classification_type", "classification_evidence", "classification_confidence", "review_required", "review_reason", "classification_source"]
name = lambda cid: CAT[cid]["category_name"] if cid in CAT else cid
rows = []
for k, it in res.items():
    flag, why = apply_review_rules(it)
    rows.append({"note_key": k, "primary_reason_id": it["primary_reason"], "primary_reason_name": name(it["primary_reason"]),
                 "primary_subcategory": it["primary_subcategory"],
                 "secondary_reason_ids": ", ".join(it["secondary_reasons"]),
                 "secondary_reason_names": ", ".join(name(s) for s in it["secondary_reasons"]),
                 "classification_type": it["classification_type"], "classification_evidence": it["evidence"],
                 "classification_confidence": float(it["confidence"]), "review_required": flag, "review_reason": why,
                 "classification_source": "llm"})
cls = pd.DataFrame(rows, columns=["note_key"] + OUT_COLS)

df = df.drop(columns=[c for c in OUT_COLS if c in df.columns]).merge(cls, on="note_key", how="left")
blank = df["processing_status"] == "BLANK"
df.loc[blank, OUT_COLS] = ["BLANK", name("BLANK"), None, "", "", "UNKNOWN", "", 1.0, False, None, "rule: blank note"]
MAIN_MAP = CONFIG["main_category_map"]
unmapped = sorted(set(CAT_IDS) - set(MAIN_MAP))
if unmapped:
    warn(f"Categories missing from CONFIG['main_category_map'] (main_category left blank): {unmapped}")
df["main_category"] = df["primary_reason_id"].map(MAIN_MAP)
DETAIL_MAP = CONFIG["detail_category_map"]
def detail_of(cid, sub):
    sub = "" if pd.isna(sub) or sub is None else str(sub)
    m = DETAIL_MAP.get(cid, {})
    if sub in m: return m[sub]
    return f"{str(cid).lower()}_" + (re.sub(r"[^a-z0-9]+", "_", sub.lower()).strip("_") or "general")   # fallback, warned below
df["detail_category"] = [detail_of(c, s) for c, s in zip(df["primary_reason_id"], df["primary_subcategory"])]
miss = sorted({(c, "" if pd.isna(s) else s) for c, s in zip(df["primary_reason_id"], df["primary_subcategory"])
               if ("" if pd.isna(s) else s) not in DETAIL_MAP.get(c, {})})
if miss:
    warn(f"Subcategories missing from CONFIG['detail_category_map'] (auto-named): {miss[:10]}")
df["taxonomy_version"] = TAXV; df["classification_prompt_version"] = PV; df["llm_model"] = MODEL
assert df["record_id"].is_unique and len(df) == len(valid) + blank.sum()
print(f"Tickets with a classification: {df['primary_reason_id'].notna().sum():,} / {len(df):,}")
display(df["primary_reason_id"].value_counts().rename_axis("primary_reason_id").reset_index(name="tickets")
        .assign(pct=lambda x: (x["tickets"] / len(df) * 100).round(1)))
df.to_pickle(ART / f"07_classified_{TAXV}.pkl")
RUN["status"]["phase7"] = "complete" if complete else f"incomplete ({len(res)}/{len(unique_notes)} notes)"

### 7b · Form fields for notes that give no reason

From 2022 the form's direction value was often copied into the note, so the note itself says nothing (FORM_DIRECTION_ONLY). The same applies to BLANK notes and to UNKNOWN notes (text but no reason); DELETED is left alone because the staff is not leaving. For those tickets the reason is read from the form's structured fields instead, following the form's decision tree (reviewer's tree, 2026-11-25). The direction is the note when it is 'Voluntary' / 'Involuntary', otherwise the form's `Reason for Disengagement` dropdown (AQ); 'For Record Purpose Only' and 'Deleted' there map straight to those categories.

* **Voluntary** → AR `Type of Voluntary Reason` (stored as `<voluntary reason>.<Type of Involuntary>`), no further sub-reason.
* **Involuntary** → AS `Type of Involuntary` → AT `Type of Customer Initiated` or AU `Type of Cloudstaff Initiated` (the branch) → that branch's own sub-reason column (e.g. Business Reasons → AY, Service Dissatisfaction → BC). "Role being moved", "Performance Issues" and "Attendance Issues" go one column deeper (BH, BD, BI). A column only accepts the values the tree lists for it, so stray values in the wrong column are ignored.
* If the branch's own column is empty, a more specific answer from another sub-reason column is used only when it stays in the same main category; otherwise the branch's default applies. With no branch recorded at all, the most specific sub-reason anywhere is used, then the sheet's derived `UB Main Reason`, then the `Disengagement Category` text. A buy-out in `Reason for Resignation` wins.

Each form value maps to an approved category and subcategory in `VOLUNTARY`, `COLUMNS` and `BRANCHES` below; edit them there. An optional third element sets `detail_category` directly, for detail the taxonomy subcategories lack (`performance_failed_review`, `downturn_downsizing`, `performance_team_conflict`); these apply to form-derived tickets only. Resolved tickets get `classification_source = "form fields"`, the form path as evidence, and no confidence score (it's a lookup, not a judgement). Evidence starts with the note-based category it replaced (e.g. `[note: BLANK]`). Tickets that can't be resolved keep their note-based category. This is the one place structured fields drive a label, at the reviewer's request (2026-11-25).

**Added after the Master Reason spot-check (2026-11-25).** A buy-out in `Reason for Resignation` counts in any spelling (Buyout, Client buy out, ...). Four more `Disengagement Category` values are read: Client Terminated, Business downturn - Budget Issues, Suspended Account, and CS Now reliever / temporary. Test tickets (note or comment is just 'test' / 'test only') become BLANK. Tickets whose comment says the ticket or disengagement was cancelled become DELETED (Cancelled / withdrawn). Both rules run last and are marked in `classification_source`.

In [ ]:
# ---- 7b. Resolve FORM_DIRECTION_ONLY, BLANK and UNKNOWN tickets from the structured form fields ----

S, A, M, R, MED = "SERVICE_DISSATISFACTION", "ATTENDANCE_AWOL", "MISCONDUCT", "RESIGNATION", "STAFF_MEDICAL"
PERF, TRAIN, FIT = "Performance / quality / productivity", "Failed training / onboarding", "Skills or role fit"
def down(col, default):                     # value that sends the reader one column deeper
    return ("->", col, default)

VOLUNTARY = {  # AR - Type of Voluntary Reason (no further sub-reason)
    "Better opportunity for advancement": (R, "Better opportunity / compensation"), "Better Offer Elsewhere": (R, "Better opportunity / compensation"),
    "Personal Reasons Undisclosed": (R, "Personal (undisclosed)"), "Family caring responsibilities": (R, "Family / caring"),
    "Family relocating": (R, "Relocation / overseas"), "Family business": (R, "Own business"), "Staff Moving Overseas": (R, "Relocation / overseas"),
    "Medical - Physical Health": (R, "Health / medical"), "Medical - Mental Health": (R, "Health / medical"), "Medical - Fear of COVID": (R, "Health / medical"),
    "Continue Schooling/Study": (R, "Career growth / studies"), "No Salary Increase": (R, "Better opportunity / compensation"),
    "Dissatisfied with client training": (R, "Dissatisfied with Cloudstaff / client / team"), "Client Mistreatment": (R, "Dissatisfied with Cloudstaff / client / team"),
    "Unhappy with TLO": (R, "Dissatisfied with Cloudstaff / client / team"), "Orchestrated move between clients": ("FOR_RECORD_PURPOSE_ONLY", "Internal transfer"),
    "RTO - Better Opportunity": (R, "Return-to-office / work arrangement"), "RTO - Personal Reasons": (R, "Return-to-office / work arrangement"),
    "RTO - Refuse Return": (R, "Return-to-office / work arrangement"), "Candidate - Shift Not Suitable": (R, "Schedule / night shift"),
    "Candidate - Employer counter offer": (R, "Better opportunity / compensation"), "Candidate - Failed Salary Negotiation": (R, "Better opportunity / compensation"),
}
COLUMNS = {   # sub-reason column -> {value: (category, subcategory) or down(...)}
    "Type of Business Reasons": {
        "Business Slow Down": ("BUSINESS_DOWNTURN", "Slowdown / reduced volume"), "Company Restructure": ("ROLE_NO_LONGER_NEEDED", "Restructure / reorganisation"),
        "Downsizing": ("BUSINESS_DOWNTURN", "Budget / cost cutting", "downturn_downsizing"), "Company Re-org": ("ROLE_NO_LONGER_NEEDED", "Restructure / reorganisation"),
        "Budget Issues": ("BUSINESS_DOWNTURN", "Budget / cost cutting"),
        "Role being moved": down("Type of Role Being Moved", ("ROLE_MOVED_ONSHORE_OR_PROVIDER", None))},
    "Type of Role Being Moved": {
        "Moving role onshore": ("ROLE_MOVED_ONSHORE_OR_PROVIDER", "Moved onshore / local hire"),
        "Moving Role to Another BPO": ("ROLE_MOVED_ONSHORE_OR_PROVIDER", "Moved to cheaper provider / competitor"),
        "Company Acquisition": ("CLIENT_CLOSURE_OR_ACQUISITION", "Sold / acquired / merged")},
    "Type of Service Dissatisfaction": {
        "Performance Issues": down("Type of Performance Issues", (S, PERF)), "Attendance Issues": down("Type of Attendance Issues", (A, None)),
        "Violation of client's code of conduct": (M, "Code of conduct / behaviour"),
        "Personnel - Disengaged due to issues with other team members": (S, None, "performance_team_conflict"),
        "Unable to meet KPIs": (S, PERF), "Failed Performance Review": (S, PERF, "performance_failed_review"), "Too Many Errors": (S, PERF), "Training Difficulties": (S, TRAIN),
        "Staff Mismatch": (S, FIT)},
    "Type of Performance Issues": {
        "Unable to meet KPIs": (S, PERF), "Inefficient": (S, PERF), "Failed Performance Review": (S, PERF, "performance_failed_review"), "Too Many Errors": (S, PERF),
        "Training Difficulties": (S, TRAIN), "Staff Mismatch": (S, FIT), "Failed PIP": (S, "Failed / declined PIP"), "Skills don't match CV": (S, FIT),
        "Need staff who are plug and play": (S, FIT), "Return to Work Order Ignored": (A, "Failed RTWO"), "Did not return from leave": (A, "NCNS / AWOL")},
    "Type of Attendance Issues": {
        "Unexplained/ Unauthorised absence": (A, "Absenteeism / tardiness"), "AWOL": (A, "NCNS / AWOL"), "Repeat NCNS": (A, "NCNS / AWOL"),
        "Repeat Late Logins": (A, "Absenteeism / tardiness"), "Repeat Early Logout": (A, "Absenteeism / tardiness"), "Transport Issues": (A, "Absenteeism / tardiness")},
    "Type of End of Role": {"Role no longer needed": ("ROLE_NO_LONGER_NEEDED", "Role eliminated"), "Role Redundant": ("ROLE_NO_LONGER_NEEDED", "Role eliminated")},
    "Type of Want to Hire Staff Direct": {"Formal Staff Buy Out": ("CLIENT_BUYOUT", "Buy-out"),
                                          "Client wants to hire them direct (Customer Black Flag)": ("CLIENT_BUYOUT", "Direct hire")},
    "Type of CS Now Contract": {"Contract ended no more work": ("ROLE_NO_LONGER_NEEDED", "Project / temp role ended"), "Formal Staff Buy Out": ("CLIENT_BUYOUT", "Buy-out")},
    "Type of Client Deal Retracted": {
        "Business Slow Down": ("BUSINESS_DOWNTURN", "Slowdown / reduced volume"), "Downsizing": ("BUSINESS_DOWNTURN", "Budget / cost cutting", "downturn_downsizing"),
        "Role being moved": down("Type of Role Being Moved", ("ROLE_MOVED_ONSHORE_OR_PROVIDER", None)),
        "Company Re-org": ("ROLE_NO_LONGER_NEEDED", "Restructure / reorganisation"), "Budget Issues": ("BUSINESS_DOWNTURN", "Budget / cost cutting"),
        "Candidate moved start date": (A, "No-show after onboarding"), "Failed Salary Negotiation": (R, "Better opportunity / compensation")},
    "Type of Staff Moving between MSAs": {"Role no longer needed": ("FOR_RECORD_PURPOSE_ONLY", "Internal transfer"),
                                          "MSA Novation Deed - Account being merged": ("FOR_RECORD_PURPOSE_ONLY", "Entity / billing change")},
    "Type of Temporary Role": {
        "Temporary replacement while another staff member away": ("ROLE_NO_LONGER_NEEDED", "Project / temp role ended"),
        "Client believes it was a 90 days trial": ("BUSINESS_REASON", "Client decided not to continue"),
        "Performance Issues": down("Type of Performance Issues", (S, PERF)), "Attendance Issues": down("Type of Attendance Issues", (A, None)),
        "Violation of client's code of conduct": (M, "Code of conduct / behaviour"),
        "Personnel - Disengaged due to issues with other team members": (S, None, "performance_team_conflict")},
    "Type of Staff AWOL": {"Return to Work Order Ignored": (A, "Failed RTWO"), "Staff Unresponsive": (A, "NCNS / AWOL"),
                           "Did not return from leave": (A, "NCNS / AWOL"), "Did not report for NHI": (A, "No-show after onboarding"),
                           "COC Violation": (M, "Code of conduct / behaviour")},
    "Type of Behavior Issues": {"COC Violation": (M, "Code of conduct / behaviour"), "Unprofessional Conduct": (M, "Code of conduct / behaviour"),
                                "Malicious Act by staff member": (M, "Code of conduct / behaviour")},
    "Type of Fraud": {"Time Manipulation": (M, "Fraud / falsification"), "Stats Manipulation": (M, "Fraud / falsification"),
                      "Falsification of Documents": (M, "Fraud / falsification"), "PCI Violation": (M, "Fraud / falsification")},
    "Type of Medical Issues": {"Did not start due to illness": (MED, "Unfit to work / prolonged sick leave"),
                               "Cloudstaffer failed medical requirements for the role": (MED, "Unfit to work / prolonged sick leave"),
                               "Death - Staff member passed away": (MED, "Deceased")},
}
BRANCHES = {   # AT Type of Customer Initiated / AU Type of Cloudstaff Initiated (or AS when it holds the branch) -> (sub-reason column, default)
    "business reasons": ("Type of Business Reasons", ("BUSINESS_REASON", None)),
    "service dissatisfaction": ("Type of Service Dissatisfaction", (S, None)),
    "client contract cancellation (all staff disengaged) - client pull-out": (None, ("CLIENT_ACCOUNT_EXIT", "Pull-out")),
    "client pullout": (None, ("CLIENT_ACCOUNT_EXIT", "Pull-out")),
    "end of role": ("Type of End of Role", ("ROLE_NO_LONGER_NEEDED", "Role eliminated")),
    "want to hire staff direct": ("Type of Want to Hire Staff Direct", ("CLIENT_BUYOUT", "Buy-out")),
    "cs now contract": ("Type of CS Now Contract", ("ROLE_NO_LONGER_NEEDED", "Project / temp role ended")),
    "client deal retracted": ("Type of Client Deal Retracted", ("BUSINESS_REASON", "Client decided not to continue")),
    "staff moving between msas": ("Type of Staff Moving between MSAs", ("FOR_RECORD_PURPOSE_ONLY", "Internal transfer")),
    "temporary role": ("Type of Temporary Role", ("ROLE_NO_LONGER_NEEDED", "Project / temp role ended")),
    "staff awol issues": ("Type of Staff AWOL", (A, "NCNS / AWOL")),
    "behavior issues": ("Type of Behavior Issues", (M, "Code of conduct / behaviour")),
    "fraud": ("Type of Fraud", (M, "Fraud / falsification")),
    "medical issues": ("Type of Medical Issues", (MED, None)),
    "performance issues - failed 3rd/ 5th month review": (None, (S, TRAIN)),
    "shift issues - refusal to work graveyard shift": (None, (R, "Schedule / night shift")),
    "defaulted client - suspension for non payment - bad debt": (None, ("NON_PAYMENT", "Non-payment")),
}
INITIATORS = {"customer initiated", "cloudstaff initiated"}
MAIN_OF = {}   # category_id -> main category; filled by the notebook (CONFIG main_category_map) before resolving
SCAN_ORDER = ["Type of Performance Issues", "Type of Attendance Issues", "Type of Staff AWOL", "Type of Fraud", "Type of Behavior Issues",
              "Type of Role Being Moved", "Type of End of Role", "Type of Want to Hire Staff Direct", "Type of CS Now Contract",
              "Type of Medical Issues", "Type of Temporary Role", "Type of Client Deal Retracted", "Type of Staff Moving between MSAs",
              "Type of Service Dissatisfaction", "Type of Business Reasons"]   # deepest first, used only when no branch is recorded
RESIGNATION_OVERRIDE = {"Client buy-out": ("CLIENT_BUYOUT", "Buy-out"), "Contract Buy Out": ("CLIENT_BUYOUT", "Buy-out")}
BUYOUT_RE = re.compile(r"buy\s*-?\s*out", re.I)    # any spelling (Buyout, Client buy out, ...): Master Reason spot-check 2026-11-25
UB_MAIN = {"Attendance": (A, None), "Performance": (S, None), "Medical": (MED, None),     # sheet's derived 'UB Main Reason', for empty branches
           "client has gone unresponsive": ("NON_PAYMENT", "Non-payment"),   # CS terminated / deactivated the client for non-payment (spot-check F48)
           "client pull out": ("CLIENT_ACCOUNT_EXIT", "Pull-out")}
CATEGORY_TEXT = {  # 'Disengagement Category' free text, last resort
    "Staff Resignation": (R, None), "Staff resignation": (R, None), "Migrating abroad": (R, "Relocation / overseas"),
    "Business decision": ("BUSINESS_REASON", None), "Business Reason": ("BUSINESS_REASON", None), "Business downturn": ("BUSINESS_DOWNTURN", None),
    "Client pull-out": ("CLIENT_ACCOUNT_EXIT", "Pull-out"), "Role has ended": ("ROLE_NO_LONGER_NEEDED", "Role eliminated"),
    "Role is being moved onshore/automated": ("ROLE_MOVED_ONSHORE_OR_PROVIDER", "Moved onshore / local hire"),
    "Non performance - client do not have enough documentation": (S, "Performance - insufficient documentation"),
    "Performance Issue": (S, PERF), "Cloudstaffer unable to meet KPIs": (S, PERF), "Performance - not fit for the role": (S, FIT),
    "Attendance and unresponsiveness": (A, None), "staff issue (attendance)": (A, None), "Attendance Issue": (A, None), "Attendance Issues": (A, None),
    "Attendance Issue/tardiness": (A, "Absenteeism / tardiness"),
    "Malicious Actions by Cloudstaffer": (M, None),
    # added after the Master Reason spot-check (2026-11-25)
    "Client Terminated": ("BUSINESS_REASON", None), "Business downturn - Budget Issues": ("BUSINESS_DOWNTURN", "Budget / cost cutting"),
    "Suspended Account": ("NON_PAYMENT", "Non-payment"),
    "Cloudstaffer is a Reliever/Temporary; under CS Now": ("ROLE_NO_LONGER_NEEDED", "Project / temp role ended"), "staff issue (misconduct)": (M, None), "Gross Misconduct": (M, None), "Medical Issue": (MED, None),
}
FORM_FIELD_MAP = {"voluntary": VOLUNTARY, "resignation": RESIGNATION_OVERRIDE, "branch": {k: v[1] for k, v in BRANCHES.items()},
                  "ub_main": UB_MAIN, "category_text": CATEGORY_TEXT,
                  **{f"column: {c}": {k: (v[2] if v[0] == "->" else v) for k, v in m.items()} for c, m in COLUMNS.items()}}   # flat view, for validation

def _v(x):
    if x is None or (isinstance(x, float) and pd.isna(x)) or pd.isna(x): return None
    x = str(x).strip(); return None if x in ("", "NA", "-") else x

def _walk(row, col, trail):
    """Read sub-reason column `col`; follow one level down where the tree says so. -> ((cat, sub), trail) or (None, trail)"""
    val = _v(row.get(col))
    entry = COLUMNS.get(col, {}).get(val)
    if entry is None:
        return None, trail
    trail = f"{trail} > {col.replace('Type of ', '')}: {val}"
    if entry[0] == "->":
        deeper, t2 = _walk(row, entry[1], trail)
        return (deeper, t2) if deeper else (entry[2], trail)
    return entry, trail

def resolve(row, M=None):
    """-> (category_id, subcategory, path, detail override or None); category None when unresolved"""
    r = _resolve(row)
    if len(r) == 4: return (r[0], r[1], r[3], r[2])            # entry carried a detail override
    return (r[0], r[1], r[2], None)

def _resolve(row):
    direction = str(row["Reason for Disengagement:"]).strip().lower()              # the note, when it is the direction value
    if direction not in ("voluntary", "involuntary"):
        direction = str(row.get("Reason for Disengagement") or "").strip().lower()   # else the form's direction dropdown (AQ)
    if direction == "for record purpose only":
        return ("FOR_RECORD_PURPOSE_ONLY", "Form value", "Form direction: For Record Purpose Only")
    if direction == "deleted":
        return ("DELETED", "Deleted", "Form direction: Deleted")
    rr = _v(row.get("Reason for Resignation"))
    if rr in RESIGNATION_OVERRIDE or (rr and BUYOUT_RE.search(rr)):
        return (*RESIGNATION_OVERRIDE.get(rr, ("CLIENT_BUYOUT", "Buy-out")), f"Reason for Resignation: {rr}")
    tv = _v(row.get("Type of Voluntary Reason")) or ""
    vol, _, as_from_tv = tv.partition(".")                     # AR is stored as '<voluntary reason>.<Type of Involuntary>'
    vol = _v(vol)
    if direction == "voluntary" and vol in VOLUNTARY:
        return (*VOLUNTARY[vol], f"Voluntary > {vol}")
    as_ = _v(row.get("Type of Involuntary")) or _v(as_from_tv)
    branch = _v(row.get("Type of Customer Initiated")) or _v(row.get("Type of Cloudstaff Initiated"))
    if branch is None and as_ and as_.lower() not in INITIATORS:
        branch = as_                                            # older form: the branch sits in Type of Involuntary itself
    path = f"{direction.title()} > {as_ or '?'}" + (f" > {branch}" if branch and branch != as_ else "")
    b = BRANCHES.get((branch or "").lower())
    if b:
        col, default = b
        if col:
            hit, trail = _walk(row, col, path)
            if hit: return (*hit, trail)
        for c in SCAN_ORDER:                                    # branch's own column empty: a more specific answer elsewhere,
            hit, trail = _walk(row, c, path)                    # but only if it stays in the branch's main category
            if hit and hit[1] and MAIN_OF.get(hit[0]) and MAIN_OF.get(hit[0]) == MAIN_OF.get(default[0]):
                return (*hit, trail + " (neighbouring column)")
        return (*default, path)
    if branch is None:                                          # no branch recorded: most specific sub-reason anywhere
        for col in SCAN_ORDER:
            hit, trail = _walk(row, col, path)
            if hit: return (*hit, trail)
        ub = _v(row.get("UB Main Reason"))
        if ub in UB_MAIN: return (*UB_MAIN[ub], f"{path} > UB Main Reason: {ub}")
    dc = _v(row.get("Disengagement Category"))
    if dc in CATEGORY_TEXT: return (*CATEGORY_TEXT[dc], f"{path} > Disengagement Category: {dc}")
    if direction == "voluntary" and vol: return (R, None, f"Voluntary > {vol} (not in the tree)")
    return (None, None, path)

_sub_ok = {c["category_id"]: set(c["subcategories"]) for c in approved["categories"]}
_bad = [(k, v) for g in FORM_FIELD_MAP.values() for k, v in g.items() if v[0] not in _sub_ok or (v[1] and v[1] not in _sub_ok[v[0]])]
assert not _bad, f"FORM_FIELD_MAP points at categories / subcategories that are not approved: {_bad}"

MAIN_OF.update(MAIN_MAP)
FORM_ROUTED = ["FORM_DIRECTION_ONLY", "BLANK", "UNKNOWN"]     # DELETED is left alone: the staff is not leaving
fdo = df["primary_reason_id"].isin(FORM_ROUTED)
was = df.loc[fdo, "primary_reason_id"].copy()
if fdo.any():
    r = df.loc[fdo].apply(resolve, axis=1, result_type="expand"); r.columns = ["cat", "sub", "path", "detail"]
    ok = r["cat"].notna()
    idx = r.index[ok]
    df.loc[idx, "primary_reason_id"] = r.loc[idx, "cat"]
    df.loc[idx, "primary_reason_name"] = r.loc[idx, "cat"].map(name)
    df.loc[idx, "primary_subcategory"] = r.loc[idx, "sub"]
    df.loc[idx, ["secondary_reason_ids", "secondary_reason_names"]] = ""
    df.loc[idx, "classification_type"] = "EXPLICIT"
    df.loc[idx, "classification_evidence"] = "[note: " + was.loc[idx] + "] " + r.loc[idx, "path"]
    df.loc[idx, "classification_confidence"] = np.nan
    df.loc[idx, "review_required"] = False; df.loc[idx, "review_reason"] = None
    df.loc[idx, "classification_source"] = "form fields"
    df["main_category"] = df["primary_reason_id"].map(MAIN_MAP)
    df["detail_category"] = [detail_of(c, s) for c, s in zip(df["primary_reason_id"], df["primary_subcategory"])]
    ov = r.loc[idx, "detail"].dropna()                         # detail overrides set by the form map (e.g. performance_failed_review)
    df.loc[ov.index, "detail_category"] = ov
    print(f"Tickets routed to the form fields: {int(fdo.sum()):,} | resolved: {int(ok.sum()):,} | kept as they were: {int((~ok).sum()):,}")
    display(pd.crosstab(r["cat"].fillna("(kept)"), was.rename("note-based category"), margins=True))
    display(r.loc[ok, "path"].value_counts().head(15).rename_axis("most common form paths").reset_index(name="tickets"))
    logger.info(f"Form fields: {int(ok.sum())} of {int(fdo.sum())} {'/'.join(FORM_ROUTED)} tickets resolved")
    RUN["status"]["phase7b"] = f"{int(ok.sum())}/{int(fdo.sum())} {'/'.join(FORM_ROUTED)} tickets resolved from form fields"
# ---- test and cancelled tickets (Master Reason spot-check 2026-11-25) --------------------------
TEST_RE = r"^\s*(?:this is a )?test(?:ing)?(?: only| ticket)?\s*[.!]*\s*$"
CANCEL_RE = r"^\s*cancel+ed\s+(?:ticket|disengagement)|cancel the (?:deactivation|disengagement)"
_txt = lambda c: df[c].astype("string") if c in df.columns else pd.Series(pd.NA, index=df.index, dtype="string")
is_test = np.logical_or.reduce([_txt(c).str.contains(TEST_RE, case=False, regex=True, na=False).to_numpy()
                                for c in (TEXT, "Comment", "Add your comments below")])
is_cancel = np.logical_or.reduce([_txt(c).str.contains(CANCEL_RE, case=False, regex=True, na=False).to_numpy()
                                  for c in ("Comment", "Add your comments below")]) & ~is_test
for mask, cid, sub, src, why in [(is_test, "BLANK", None, "rule: test ticket", "test ticket (note or comment says 'test')"),
                                 (is_cancel, "DELETED", "Cancelled / withdrawn", "rule: cancelled ticket", "comment says the ticket / disengagement was cancelled")]:
    idx = df.index[mask]
    if len(idx):
        df.loc[idx, "classification_evidence"] = "[was: " + df.loc[idx, "primary_reason_id"].astype(str) + "] " + why
        df.loc[idx, ["primary_reason_id", "primary_subcategory", "classification_source"]] = [cid, sub, src]
        df.loc[idx, "primary_reason_name"] = name(cid)
        df.loc[idx, ["secondary_reason_ids", "secondary_reason_names"]] = ""
        df.loc[idx, "classification_type"] = "EXPLICIT"; df.loc[idx, "classification_confidence"] = 1.0
        df.loc[idx, "review_required"] = False; df.loc[idx, "review_reason"] = None
print(f"Rules: {int(is_test.sum())} test tickets -> BLANK, {int(is_cancel.sum())} cancelled tickets -> DELETED")
df["main_category"] = df["primary_reason_id"].map(MAIN_MAP)
_ov = df["detail_category"].copy()
df["detail_category"] = [detail_of(c, s_) for c, s_ in zip(df["primary_reason_id"], df["primary_subcategory"])]
_keep = df["classification_source"].eq("form fields") & _ov.notna()          # keep the form map's detail overrides
df.loc[_keep, "detail_category"] = _ov[_keep]
df.to_pickle(ART / f"07_classified_{TAXV}.pkl")


## Phase 8 · Human validation sample and metrics

Sampling is done **by distinct note**, because a ticket-level sample would be mostly "Involuntary" repeated. About 500 notes are stratified across every primary category (rare ones included), confidence high and low, multi-reason, UNKNOWN, outliers (-1), large and small clusters, and review-flagged. One ticket represents each note, and `tickets_with_same_note` shows how far a verdict carries.

The sample is written **once per taxonomy version** to `outputs/validation_<version>.xlsx`, and a rerun never overwrites it. Reviewers fill `human_primary_reason`, `human_secondary_reasons`, `correct` and `comments`. Metrics are computed only from reviewed rows. Operational metrics are always shown.

Two scores are reported. **Accuracy (exact match)** needs the predicted primary to equal the reviewer's. **Accuracy (reviewer rules)** also accepts the reviewer-agreed allowances in CONFIG: a performance + attendance note is right when one label is primary and the other secondary (`validation_equivalent_pairs`), and ROLE_NO_LONGER_NEEDED counts as BUSINESS_REASON at large (`validation_rollup`). Per-category precision / recall stay exact-match.

The sample above was used to write the taxonomy rules, so its score flatters the classifier. A **fresh spot-check** (`outputs/spotcheck_<version>.xlsx`, `spotcheck_size` random notes that were never in any validation file) gives the honest estimate. It's reviewed **blind**: the predictions sit in hidden columns at the right, so pick the category before unhiding them.

Both reviewed files are combined into a **gold dataset** (`outputs/gold_labels.xlsx` and `.csv`), rebuilt on every run so later label edits flow in. It holds the reviewer's labels plus the current model's answers in separate `model_*` columns, so the labels stay fixed when the taxonomy changes.

The reviewed labels are the only ground truth here. The form's structured fields aren't used as truth, because they're unverified before April 2024.

In [ ]:
rng = np.random.default_rng(CONFIG["random_seed"])
note_view = (df[df["processing_status"] == "VALID_TEXT"].groupby("note_key")
             .agg(record_id=("record_id", "first"), tickets=("record_id", "size"), cluster_id=("cluster_id", "first"),
                  primary=("primary_reason_id", "first"), secondary=("secondary_reason_ids", "first"),
                  conf=("classification_confidence", "first"), review=("review_required", "first"))
             .dropna(subset=["primary"]).reset_index())
csize = note_view.groupby("cluster_id")["tickets"].transform("sum")
note_view["stratum_flags"] = ""
def take(mask, n, label):
    pool = note_view[mask & ~note_view["note_key"].isin(chosen)]
    if len(pool) == 0: return
    pick = pool.sample(n=min(n, len(pool)), random_state=int(rng.integers(1_000_000_000)))
    for k in pick["note_key"]:
        chosen[k] = chosen.get(k, label)
chosen = {}
if complete and not (OUT / f"validation_{TAXV}.xlsx").exists():      # only needed when a new sample has to be drawn
    N = CONFIG["validation_sample_size"]
    for cat_id, g in note_view.groupby("primary"):                       # every category, rare ones fully covered up to 12
        take(note_view["primary"].eq(cat_id) & note_view["conf"].ge(0.85), 6, f"category:{cat_id} high-conf")
        take(note_view["primary"].eq(cat_id), 6, f"category:{cat_id}")
    take(note_view["conf"] < CONFIG["confidence_threshold"], 50, "low confidence")
    take(note_view["secondary"].fillna("").astype(str).str.len() > 0, 30, "multi-reason")
    take(note_view["primary"].eq("UNKNOWN"), 20, "unknown")
    take(note_view["cluster_id"].eq(-1), 40, "outlier (-1)")
    take(note_view["review"].astype(bool), 30, "review flagged")
    take(csize >= csize.quantile(0.75), 25, "large cluster")
    take(csize <= csize.quantile(0.25), 25, "small cluster")
    take(note_view["conf"] >= 0.9, 20, "high confidence")
    if len(chosen) < N:                                                 # fill, weighted by ticket volume
        rest = note_view[~note_view["note_key"].isin(chosen)]
        n_fill = min(N - len(chosen), len(rest)); w = np.sqrt(rest["tickets"])
        w = w.clip(upper=w.sum() / max(n_fill, 1))                     # no single note may need an inclusion probability above 1
        pick = rest.sample(n=n_fill, weights=w, random_state=CONFIG["random_seed"])
        for k in pick["note_key"]: chosen[k] = "volume-weighted fill"
    print(f"Validation sample: {len(chosen)} distinct notes")

In [ ]:
VAL_XLSX = OUT / f"validation_{TAXV}.xlsx"
VAL_COLS = ["validation_id", "sample_stratum", "record_id", "tickets_with_same_note", "cluster_id", "note_original", "note_clean",
            "predicted_primary_reason", "predicted_subcategory", "predicted_secondary_reasons", "predicted_type", "predicted_confidence",
            "predicted_evidence", "review_required", "human_primary_reason", "human_secondary_reasons", "correct", "reviewer_comments"]
HUMAN_COLS = ["human_primary_reason", "human_secondary_reasons", "correct", "reviewer_comments"]
def previous_reviewed_sample():
    # the most recent validation file of an EARLIER taxonomy version that holds reviewer labels
    for pv in sorted((p for p in OUT.glob("validation_*.xlsx") if p != VAL_XLSX and not p.name.startswith("~$")),
                     key=lambda p: p.stat().st_mtime, reverse=True):
        d = pd.read_excel(pv, "Validation", dtype=object)
        if d["human_primary_reason"].notna().any():
            return d, pv.name
    return None, None

carried = None
if complete and not VAL_XLSX.exists():
    carried, carried_from = previous_reviewed_sample()
    if carried is not None:
        # Reuse the reviewed notes so a new taxonomy version is scored on the same ground truth.
        # 'correct' judged the OLD prediction, so it is cleared; the human labels carry over as they are.
        carried = carried.assign(record_id=carried["record_id"].astype(str), _order=range(len(carried)))
        s = df.assign(record_id=df["record_id"].astype(str)).merge(
            carried[["record_id", "sample_stratum", "_order"]], on="record_id").merge(note_view[["note_key", "tickets"]], on="note_key")
        s = s.sort_values("_order")
        print(f"Reusing the {len(s)} reviewed notes from {carried_from} (labels carried over, 'correct' cleared).")
    else:
        s = df[df["note_key"].isin(chosen)].drop_duplicates("note_key").copy()
        s["sample_stratum"] = s["note_key"].map(chosen)
        s = s.merge(note_view[["note_key", "tickets"]], on="note_key").sort_values(["sample_stratum", "primary_reason_id"])
    val = pd.DataFrame({
        "validation_id": [f"V{i:04d}" for i in range(1, len(s) + 1)], "sample_stratum": s["sample_stratum"].values,
        "record_id": s["record_id"].values, "tickets_with_same_note": s["tickets"].values, "cluster_id": s["cluster_id"].values,
        "note_original": s[TEXT].astype(str).str.replace("_x000D_", "", regex=False).values, "note_clean": s["note_clean"].values,
        "predicted_primary_reason": s["primary_reason_id"].values, "predicted_subcategory": s["primary_subcategory"].values,
        "predicted_secondary_reasons": s["secondary_reason_ids"].values, "predicted_type": s["classification_type"].values,
        "predicted_confidence": s["classification_confidence"].values, "predicted_evidence": s["classification_evidence"].values,
        "review_required": s["review_required"].values,
        "human_primary_reason": "", "human_secondary_reasons": "", "correct": "", "reviewer_comments": ""})
    if carried is not None:
        lab = carried.set_index("record_id")
        for col in ("human_primary_reason", "human_secondary_reasons", "reviewer_comments"):
            val[col] = val["record_id"].astype(str).map(lab[col]).where(lambda x: x.notna(), "")
    with pd.ExcelWriter(VAL_XLSX, engine="xlsxwriter") as xw:
        val.to_excel(xw, sheet_name="Validation", index=False)
        pd.DataFrame({"category_id": CAT_IDS, "category_name": [name(c) for c in CAT_IDS]}).to_excel(xw, sheet_name="Categories", index=False)
        wb = xw.book; ws = xw.sheets["Validation"]
        wrap = wb.add_format({"text_wrap": True, "valign": "top"}); edit = wb.add_format({"text_wrap": True, "valign": "top", "bg_color": "#FFF7D6"})
        for i, c in enumerate(VAL_COLS):
            w = 60 if c in ("note_original",) else 35 if c in ("predicted_evidence", "reviewer_comments", "note_clean") else 22
            ws.set_column(i, i, w, edit if c.startswith("human_") or c in ("correct", "reviewer_comments") else wrap)
        ws.set_column(VAL_COLS.index("note_clean"), VAL_COLS.index("note_clean"), 35, wrap, {"hidden": True})
        ws.freeze_panes(1, 3); ws.autofilter(0, 0, len(val), len(VAL_COLS) - 1)
        hp = VAL_COLS.index("human_primary_reason"); cc = VAL_COLS.index("correct")
        ws.data_validation(1, hp, len(val), hp, {"validate": "list", "source": f"=Categories!$A$2:$A${len(CAT_IDS) + 1}", "ignore_blank": True})
        ws.data_validation(1, cc, len(val), cc, {"validate": "list", "source": ["Y", "N"], "ignore_blank": True})
        xw.sheets["Categories"].set_column(0, 1, 40)
    print(f"Validation sample written: {VAL_XLSX}")
elif VAL_XLSX.exists():
    print(f"{VAL_XLSX.name} exists - kept (it may hold reviewer labels).")
val = pd.read_excel(VAL_XLSX, "Validation", dtype=object) if VAL_XLSX.exists() else pd.DataFrame(columns=VAL_COLS)

In [ ]:
# ---- 8b. Fresh blind spot-check ------------------------------------------------
# Simple random sample of unique notes that were never in ANY validation file, so the rules were not tuned on them.
SPOT_XLSX = OUT / f"spotcheck_{TAXV}.xlsx"
SPOT_COLS = ["spot_id", "record_id", "tickets_with_same_note", "note_original", "human_primary_reason", "human_secondary_reasons",
             "reviewer_comments", "predicted_primary_reason", "predicted_secondary_reasons", "predicted_confidence"]
def keys_in_validation_files():
    ids = set()
    for pv in OUT.glob("validation_*.xlsx"):
        if not pv.name.startswith("~$"):
            ids |= set(pd.read_excel(pv, "Validation", dtype=object)["record_id"].astype(str))
    return set(df.loc[df["record_id"].astype(str).isin(ids), "note_key"])

if complete and CONFIG.get("spotcheck_size") and not SPOT_XLSX.exists():
    pool = note_view[~note_view["note_key"].isin(keys_in_validation_files())]
    pick = pool.sample(n=min(CONFIG["spotcheck_size"], len(pool)), random_state=CONFIG["random_seed"] + 1)
    first = df.drop_duplicates("note_key").set_index("note_key")
    spot = pd.DataFrame({
        "spot_id": [f"S{i:03d}" for i in range(1, len(pick) + 1)], "record_id": pick["record_id"].values,
        "tickets_with_same_note": pick["tickets"].values,
        "note_original": first.loc[pick["note_key"], TEXT].astype(str).str.replace("_x000D_", "", regex=False).values,
        "human_primary_reason": "", "human_secondary_reasons": "", "reviewer_comments": "",
        "predicted_primary_reason": pick["primary"].values, "predicted_secondary_reasons": pick["secondary"].fillna("").values,
        "predicted_confidence": pick["conf"].values})
    with pd.ExcelWriter(SPOT_XLSX, engine="xlsxwriter") as xw:
        spot.to_excel(xw, sheet_name="Spot check", index=False)
        pd.DataFrame([{"category_id": c, "category_name": name(c), "definition": CAT[c]["definition"]} for c in CAT_IDS]) \
          .to_excel(xw, sheet_name="Categories", index=False)
        wb = xw.book; ws = xw.sheets["Spot check"]
        wrap = wb.add_format({"text_wrap": True, "valign": "top"}); edit = wb.add_format({"text_wrap": True, "valign": "top", "bg_color": "#FFF7D6"})
        for i, c in enumerate(SPOT_COLS):
            w = 80 if c == "note_original" else 35 if c.startswith("human_") or c == "reviewer_comments" else 14
            ws.set_column(i, i, w, edit if c.startswith("human_") or c == "reviewer_comments" else wrap,
                          {"hidden": c.startswith("predicted_")})                      # blind review: unhide only after labelling
        hp = SPOT_COLS.index("human_primary_reason")
        ws.data_validation(1, hp, len(spot), hp, {"validate": "list", "source": f"=Categories!$A$2:$A${len(CAT_IDS) + 1}", "ignore_blank": True})
        ws.freeze_panes(1, 4)
        xs = xw.sheets["Categories"]; xs.set_column(0, 1, 32); xs.set_column(2, 2, 100, wrap)
    print(f"Spot-check written: {SPOT_XLSX} ({len(spot)} notes, predictions hidden)")
elif SPOT_XLSX.exists():
    print(f"{SPOT_XLSX.name} exists - kept (it may hold reviewer labels).")
spot = pd.read_excel(SPOT_XLSX, "Spot check", dtype=object) if SPOT_XLSX.exists() else pd.DataFrame(columns=SPOT_COLS)


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.preprocessing import MultiLabelBinarizer

def operational_metrics(d):
    n = len(d); nb = d[d["processing_status"] == "VALID_TEXT"]
    rows = [("Total tickets", n, 100.0),
            ("BLANK", int((d["primary_reason_id"] == "BLANK").sum()), None),
            ("UNKNOWN / unclassified", int((d["primary_reason_id"] == "UNKNOWN").sum()), None),
            ("Flagged for review", int(d["review_required"].fillna(False).astype(bool).sum()), None),
            ("EXPLICIT (non-blank notes)", int((nb["classification_type"] == "EXPLICIT").sum()), None),
            ("INFERRED (non-blank notes)", int((nb["classification_type"] == "INFERRED").sum()), None),
            ("UNKNOWN type (non-blank notes)", int((nb["classification_type"] == "UNKNOWN").sum()), None),
            ("With at least one secondary reason", int((d["secondary_reason_ids"].fillna("") != "").sum()), None)]
    out = pd.DataFrame(rows, columns=["metric", "tickets", "pct"])
    out["pct"] = (out["tickets"] / n * 100).round(1)
    return out

def split_ids(v):
    return sorted({x.strip() for x in str(v).replace(";", ",").split(",") if x.strip() and x.strip().lower() != "nan"})

EQ_PAIRS = [frozenset(x) for x in CONFIG.get("validation_equivalent_pairs", [])]
ROLLUP = CONFIG.get("validation_rollup", {})
def reviewer_rule_match(h, p, sec):
    # exact match, or a reviewer-agreed allowance (see CONFIG)
    if h == p or ROLLUP.get(h, h) == ROLLUP.get(p, p):
        return True
    return frozenset((h, p)) in EQ_PAIRS and h in sec

def rule_scores(rev, prefix=""):
    h, p = rev["human_primary_reason"].tolist(), rev["predicted_primary_reason"].tolist()
    sec = rev["predicted_secondary_reasons"].map(split_ids).tolist()
    exact = np.array([a == b for a, b in zip(h, p)]); rule = np.array([reviewer_rule_match(a, b, s) for a, b, s in zip(h, p, sec)])
    anyl = rule | np.array([a in s for a, s in zip(h, sec)])
    mm = CONFIG["main_category_map"]
    main = np.array([mm.get(a, a) == mm.get(b, b) for a, b in zip(h, p)])
    return [(f"{prefix}Accuracy (exact match)", round(exact.mean(), 3)), (f"{prefix}Accuracy (reviewer rules)", round(rule.mean(), 3)),
            (f"{prefix}Reviewer label anywhere in primary / secondary", round(anyl.mean(), 3)),
            (f"{prefix}Accuracy (main category)", round(main.mean(), 3))], rule

def spot_metrics(s):
    s = s.copy()
    for c in ("human_primary_reason", "predicted_secondary_reasons"):
        s[c] = s[c].fillna("").astype(str).str.strip()
    rev = s[s["human_primary_reason"] != ""]
    if rev.empty:
        return None
    rows, rule = rule_scores(rev, "Fresh spot-check: ")
    w = pd.to_numeric(rev["tickets_with_same_note"], errors="coerce").fillna(1).to_numpy()
    rows = [("Fresh spot-check: reviewed notes", len(rev))] + rows + \
           [("Fresh spot-check: Accuracy (reviewer rules), ticket-weighted", round(float((rule * w).sum() / w.sum()), 3))]
    return pd.DataFrame(rows, columns=["metric", "value"])

def validation_metrics(v):
    v = v.copy()
    for c in ("correct", "human_primary_reason", "human_secondary_reasons", "predicted_secondary_reasons"):
        v[c] = v[c].fillna("").astype(str).str.strip()          # pandas 3 keeps NaN through astype(str), so fill first
    v["correct"] = v["correct"].str.upper()
    v.loc[(v["human_primary_reason"] == "") & (v["correct"] == "Y"), "human_primary_reason"] = v["predicted_primary_reason"]
    rev = v[v["human_primary_reason"] != ""]
    if rev.empty:
        return None
    y, p = rev["human_primary_reason"], rev["predicted_primary_reason"]
    labels = sorted(set(y) | set(p))
    P, R, F, S = precision_recall_fscore_support(y, p, labels=labels, zero_division=0)
    per_cat = pd.DataFrame({"category": labels, "precision": P.round(3), "recall": R.round(3), "f1": F.round(3), "support (human)": S})
    mp, mr, mf, _ = precision_recall_fscore_support(y, p, average="macro", zero_division=0)
    rule_rows, _ = rule_scores(rev)
    summary = pd.DataFrame([("Reviewed rows", len(rev))] + rule_rows + [
                            ("Macro precision", round(mp, 3)), ("Macro recall", round(mr, 3)), ("Macro F1", round(mf, 3))], columns=["metric", "value"])
    cm = pd.DataFrame(confusion_matrix(y, p, labels=labels), index=[f"human:{l}" for l in labels], columns=[f"pred:{l}" for l in labels])
    hs = rev["human_secondary_reasons"].map(split_ids); ps = rev["predicted_secondary_reasons"].map(split_ids)
    use = (rev["correct"] == "Y") | (rev["human_secondary_reasons"] != "")
    hs = hs.where(~((rev["correct"] == "Y") & (rev["human_secondary_reasons"] == "")), ps)
    if use.any():
        mlb = MultiLabelBinarizer(classes=CAT_IDS).fit([CAT_IDS])
        sp, sr, sf, _ = precision_recall_fscore_support(mlb.transform(hs[use]), mlb.transform(ps[use]), average="micro", zero_division=0)
        summary = pd.concat([summary, pd.DataFrame([("Secondary reasons: micro precision", round(sp, 3)), ("Secondary reasons: micro recall", round(sr, 3)),
                                                    ("Secondary reasons: micro F1", round(sf, 3))], columns=["metric", "value"])])
    return {"summary": summary, "per_category": per_cat, "confusion": cm}

ops = operational_metrics(df); display(ops)
vm = validation_metrics(val) if len(val) else None
sm = spot_metrics(spot) if len(spot) else None
if vm and sm is not None:
    vm["summary"] = pd.concat([vm["summary"], sm], ignore_index=True)
if vm:
    display(vm["summary"]); display(vm["per_category"])
else:
    print(f"No reviewed validation rows yet - fill {VAL_XLSX.name} and rerun to get accuracy / precision / recall / F1.")
conf_hist = pd.cut(df.loc[df["processing_status"] == "VALID_TEXT", "classification_confidence"],
                   bins=[0, .5, .6, .7, .75, .8, .9, .95, 1.0], include_lowest=True).value_counts().sort_index()
RUN["status"]["phase8"] = (f"validation sample {len(val)} notes; reviewed: {0 if not vm else int(vm['summary'].iloc[0, 1])}; "
                          f"spot-check {len(spot)} notes; reviewed: {0 if sm is None else int(sm.iloc[0, 1])}")

In [ ]:
# ---- 8c. Gold dataset: every human-labelled note in one place --------------------
# Validation sample (stratified; it was used to write the rules) + fresh blind spot-check (simple random).
# gold_* columns are the reviewer's labels and never change with the taxonomy; model_* columns are the CURRENT version's answers.
GOLD_XLSX = OUT / "gold_labels.xlsx"; GOLD_CSV = OUT / "gold_labels.csv"
GOLD_COLS = ["gold_id", "source", "source_id", "sample_stratum", "record_id", "note_key", "tickets_with_same_note", "note_original", "note_clean",
             "gold_primary_reason", "gold_secondary_reasons", "gold_main_category", "reviewer_comments",
             "model_primary_reason", "model_secondary_reasons", "model_main_category", "model_detail_category", "model_matches_primary", "model_matches_main", "model_taxonomy_version"]

def build_gold(v, s):
    parts = []
    if len(v):
        parts.append(v.assign(source="validation sample", source_id=v["validation_id"]))
    if len(s):
        parts.append(s.assign(source="fresh spot-check (blind)", source_id=s["spot_id"], sample_stratum="simple random"))
    if not parts:
        return pd.DataFrame(columns=GOLD_COLS)
    g = pd.concat(parts, ignore_index=True)
    g["gold_primary_reason"] = g["human_primary_reason"].fillna("").astype(str).str.strip()
    g = g[g["gold_primary_reason"] != ""].copy()
    bad = sorted(set(g["gold_primary_reason"]) - set(CAT_IDS))
    if bad:
        warn(f"Gold labels that are not approved category ids: {bad}")
    tix = df.assign(record_id=df["record_id"].astype(str)).set_index("record_id")
    g["record_id"] = g["record_id"].astype(str)
    g["note_key"] = g["record_id"].map(tix["note_key"])
    dup = g["note_key"].duplicated(keep="first")
    if dup.any():
        warn(f"{int(dup.sum())} notes are labelled in both files; the validation-sample label is kept")
        g = g[~dup]
    g["note_clean"] = g["record_id"].map(tix["note_clean"])
    g["note_original"] = g["record_id"].map(tix[TEXT]).astype(str).str.replace("_x000D_", "", regex=False)
    g["tickets_with_same_note"] = g["note_key"].map(note_view.set_index("note_key")["tickets"])
    g["gold_secondary_reasons"] = g["human_secondary_reasons"].map(lambda x: ", ".join(split_ids(x)))
    g["gold_main_category"] = g["gold_primary_reason"].map(MAIN_MAP)
    g["reviewer_comments"] = g["reviewer_comments"].fillna("").astype(str)
    g["model_primary_reason"] = g["record_id"].map(tix["primary_reason_id"])
    g["model_secondary_reasons"] = g["record_id"].map(tix["secondary_reason_ids"]).fillna("")
    g["model_main_category"] = g["model_primary_reason"].map(MAIN_MAP)
    g["model_detail_category"] = g["record_id"].map(tix["detail_category"])
    g["model_matches_primary"] = g["model_primary_reason"] == g["gold_primary_reason"]
    g["model_matches_main"] = g["model_main_category"] == g["gold_main_category"]
    g["model_taxonomy_version"] = TAXV
    g["gold_id"] = [f"G{i:04d}" for i in range(1, len(g) + 1)]
    return g[GOLD_COLS].reset_index(drop=True)

gold = build_gold(val, spot)
if len(gold):
    about = pd.DataFrame([
        ("What", "Every note a human has labelled: the stratified validation sample plus the fresh blind spot-check. One row per distinct note."),
        ("gold_* columns", "The reviewer's labels. They do not change when the taxonomy or the model changes."),
        ("gold_main_category", "gold_primary_reason mapped with CONFIG['main_category_map']."),
        ("model_* columns", f"The model's answers for taxonomy {TAXV}, refreshed every run, for comparison only."),
        ("source", "'validation sample' was stratified (rare categories over-sampled) and used to write the taxonomy rules, so it flatters the model. "
                   "'fresh spot-check (blind)' is a simple random sample of notes the rules were never tuned on: use it for honest accuracy."),
        ("Editing labels", "Edit the source files (validation_<version>.xlsx, spotcheck_<version>.xlsx), not this file; it is rebuilt on every run."),
        ("Built", dt.datetime.now().isoformat(timespec="seconds")),
    ], columns=["item", "detail"])
    with pd.ExcelWriter(GOLD_XLSX, engine="xlsxwriter") as xw:
        gold.to_excel(xw, sheet_name="Gold", index=False); about.to_excel(xw, sheet_name="About", index=False)
        wb = xw.book; wrap = wb.add_format({"text_wrap": True, "valign": "top"}); ws = xw.sheets["Gold"]
        for i, c in enumerate(GOLD_COLS):
            ws.set_column(i, i, 70 if c == "note_original" else 30 if c.startswith(("gold_", "model_")) or c == "reviewer_comments" else 14,
                          wrap if c == "note_original" else None, {"hidden": c == "note_clean"})
        ws.freeze_panes(1, 3); ws.autofilter(0, 0, len(gold), len(GOLD_COLS) - 1)
        xw.sheets["About"].set_column(0, 0, 22); xw.sheets["About"].set_column(1, 1, 120, wrap)
    gold.to_csv(GOLD_CSV, index=False, encoding="utf-8-sig")
    summary_gold = (gold.groupby("source").agg(notes=("gold_id", "size"), tickets=("tickets_with_same_note", "sum"),
                                               model_matches_primary=("model_matches_primary", "mean"), model_matches_main=("model_matches_main", "mean"))
                    .round(3))
    print(f"Gold dataset: {len(gold)} labelled notes -> {GOLD_XLSX.name} / {GOLD_CSV.name}")
    display(summary_gold)
    display(gold["gold_main_category"].value_counts().rename_axis("gold_main_category").reset_index(name="notes"))
RUN["status"]["gold"] = f"{len(gold)} labelled notes"


### 8d · Blind spot-check of the form-derived labels

Step 7b labels about half the tickets from the form fields, and the form fields are unverified before April 2024. The validation sample and the first spot-check only cover note-based labels, so this check covers the form-derived ones.

`form_spotcheck_size` tickets are drawn at random from the form-derived tickets that also carry **free text the reviewer can judge from** (`Comment`, `Add your comments below` or the summary fields): 2,121 of 3,824 tickets on 25 Nov 2026. The score applies to that group. The form path and the predicted labels sit in hidden columns. Fill `human_main_category` (required) and `human_primary_reason` (optional), or pick `CANT_TELL`, then unhide the columns.

The file `outputs/spotcheck_form_<version>.xlsx` is written once and never overwritten. Scores appear below once rows are labelled.


In [ ]:
# ---- 8d. Blind spot-check of form-derived labels ------------------------------------
FORM_SPOT_XLSX = OUT / f"spotcheck_form_{TAXV}.xlsx"
FS_TEXT = ["Comment", "Add your comments below", "Please provide a brief summary",
           "Please provide a brief summary of what the Cloudstaffer did well and also where they need to improve."]
FS_FORM_TEXT = ["Reason for Resignation", "Disengagement Category"]          # form text the reviewer may also use
MAIN_NAMES = sorted(set(CONFIG["main_category_map"].values()))
def _has_text(s):
    s = s.astype("string").str.strip()
    return (s.notna() & ~s.str.lower().isin(["", "nan", "none", "-", "0", "test", "n/a", "na"])).fillna(False).to_numpy(bool)

fd = df[df["classification_source"] == "form fields"]
fd_pool = fd[np.logical_or.reduce([_has_text(fd[c]) for c in FS_TEXT if c in fd.columns])]
print(f"Form-derived tickets: {len(fd):,}; with free text to judge from: {len(fd_pool):,}")
if CONFIG.get("form_spotcheck_size") and len(fd_pool) and not FORM_SPOT_XLSX.exists():
    pick = fd_pool.sample(n=min(CONFIG["form_spotcheck_size"], len(fd_pool)), random_state=CONFIG["random_seed"] + 2)
    clean = lambda s: s.astype("string").str.replace("_x000D_", "", regex=False).str.strip().fillna("")
    fs = pd.DataFrame({"spot_id": [f"F{i:02d}" for i in range(1, len(pick) + 1)], "record_id": pick["record_id"].astype(str).values,
                       "year": pd.to_datetime(pick[CONFIG["date_col"]], errors="coerce").dt.year.values,
                       "note": clean(pick[TEXT]).values})
    for c in FS_TEXT + FS_FORM_TEXT:
        fs[c] = clean(pick[c]).values if c in pick.columns else ""
    fs = fs.assign(human_main_category="", human_primary_reason="", reviewer_comments="",
                   predicted_main_category=pick["main_category"].values, predicted_primary_reason=pick["primary_reason_id"].values,
                   predicted_detail_category=pick["detail_category"].values,
                   form_path=pick["classification_evidence"].astype(str).str.replace(r"^\[note: [A-Z_]+\] ", "", regex=True).values)
    with pd.ExcelWriter(FORM_SPOT_XLSX, engine="xlsxwriter") as xw:
        fs.to_excel(xw, sheet_name="Form spot check", index=False)
        lists = pd.DataFrame({"main_category": pd.Series(MAIN_NAMES + ["CANT_TELL"]), "category_id": pd.Series(CAT_IDS + ["CANT_TELL"])})
        lists.to_excel(xw, sheet_name="Lists", index=False)
        pd.DataFrame([{"category_id": c, "main_category": CONFIG["main_category_map"].get(c), "definition": CAT[c]["definition"]} for c in CAT_IDS]) \
          .to_excel(xw, sheet_name="Categories", index=False)
        wb = xw.book; ws = xw.sheets["Form spot check"]
        wrap = wb.add_format({"text_wrap": True, "valign": "top"}); edit = wb.add_format({"text_wrap": True, "valign": "top", "bg_color": "#FFF7D6"})
        for i, c in enumerate(fs.columns):
            human = c.startswith("human_") or c == "reviewer_comments"
            w = 50 if c in ("note", "Comment", "Add your comments below") else 30 if human or c in FS_TEXT + FS_FORM_TEXT else 10
            ws.set_column(i, i, w, edit if human else wrap, {"hidden": c.startswith("predicted_") or c == "form_path"})   # blind review
        n = len(fs); cols = list(fs.columns)
        ws.data_validation(1, cols.index("human_main_category"), n, cols.index("human_main_category"),
                           {"validate": "list", "source": f"=Lists!$A$2:$A${len(MAIN_NAMES) + 2}"})
        ws.data_validation(1, cols.index("human_primary_reason"), n, cols.index("human_primary_reason"),
                           {"validate": "list", "source": f"=Lists!$B$2:$B${len(CAT_IDS) + 2}"})
        ws.freeze_panes(1, 4)
        xw.sheets["Categories"].set_column(0, 1, 30); xw.sheets["Categories"].set_column(2, 2, 100, wrap)
    print(f"Form spot-check written: {FORM_SPOT_XLSX} ({len(fs)} tickets, predictions and form path hidden)")
elif FORM_SPOT_XLSX.exists():
    print(f"{FORM_SPOT_XLSX.name} exists - kept (it may hold reviewer labels).")

form_spot = pd.read_excel(FORM_SPOT_XLSX, "Form spot check", dtype=object) if FORM_SPOT_XLSX.exists() else pd.DataFrame()
if len(form_spot):
    fsx = form_spot.copy()
    for c in ("human_main_category", "human_primary_reason"):
        fsx[c] = fsx[c].fillna("").astype(str).str.strip()
    lab = fsx[(fsx["human_main_category"] != "") & (fsx["human_main_category"] != "CANT_TELL")]
    if len(lab):
        sub = lab[(lab["human_primary_reason"] != "") & (lab["human_primary_reason"] != "CANT_TELL")]
        rows = [("Form spot-check: labelled", len(lab)), ("Form spot-check: CANT_TELL", int((fsx["human_main_category"] == "CANT_TELL").sum())),
                ("Form spot-check: Accuracy (main category)", round(float((lab["human_main_category"] == lab["predicted_main_category"]).mean()), 3))]
        if len(sub):
            rows += [("Form spot-check: sub-category labelled", len(sub)),
                     ("Form spot-check: Accuracy (sub-category, exact)", round(float((sub["human_primary_reason"] == sub["predicted_primary_reason"]).mean()), 3))]
        form_spot_metrics = pd.DataFrame(rows, columns=["metric", "value"]); display(form_spot_metrics)
        miss = lab[lab["human_main_category"] != lab["predicted_main_category"]]
        if len(miss):
            display(miss[["spot_id", "record_id", "human_main_category", "predicted_main_category", "predicted_primary_reason", "form_path", "reviewer_comments"]])
    else:
        print(f"No labels yet - fill {FORM_SPOT_XLSX.name} and rerun.")
RUN["status"]["phase8d"] = f"form spot-check {len(form_spot)} tickets"


## Phase 9 · Excel workbook

One workbook, `outputs/ub_reason_classification.xlsx`, rebuilt on every run. It contains:
* **Classified Tickets**: every original column plus the classification columns (`main_category`, `primary_reason_id`, `primary_subcategory`, `detail_category`, `classification_source`, evidence, confidence, review flags).
* **Approved Taxonomy** and **Cluster Analysis**.
* **Summary Dashboard**, with native Excel charts.
* **Validation**: a mirror of the validation file, which is where reviewers edit.
* **Cluster vs Category**: how each cluster's tickets were actually classified.
* **Revisit list** and **Taxonomy history**.
* **Profiling** and **Run Metadata**.

The validation, spot-check and gold files are separate files in `outputs/` (Phase 8). The categorized copy of the source workbook isn't built here (see open question 8).

In [ ]:
ILLEGAL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")
def xl_safe(frame):
    f = frame.copy()
    for c in f.columns:
        if f[c].dtype == object or str(f[c].dtype).startswith("string"):
            f[c] = f[c].map(lambda v: ILLEGAL.sub("", v)[:32000] if isinstance(v, str) else v)
    return f

tickets = df.drop(columns=["note_key"]).rename(columns={"note_kind": "note_kind (EDA)"})
first = list(df.columns[:df.columns.get_loc("note_clean")])
lead = [c for c in first if c != "note_kind"]
new_cols = ["note_clean", "processing_status", "cluster_id"] + OUT_COLS + ["taxonomy_version", "classification_prompt_version", "llm_model"]
new_cols.insert(new_cols.index("primary_reason_name") + 1, "main_category")
new_cols.insert(new_cols.index("primary_subcategory") + 1, "detail_category")
tickets = tickets[[c for c in lead if c in tickets.columns] + new_cols + ["note_kind (EDA)"]]

n_by = df["primary_reason_id"].value_counts()
tax_sheet = pd.DataFrame([{
    "category_id": c["category_id"], "category_name": c["category_name"], "main_category": MAIN_MAP.get(c["category_id"], ""),
    "definition": c["definition"],
    "inclusion_rules": " | ".join(c["inclusion_rules"]), "exclusion_rules": " | ".join(c["exclusion_rules"]),
    "examples": " | ".join(c["example_notes"][:6]), "subcategories": " | ".join(c["subcategories"]),
    "baseline_or_new": c["origin"], "supporting_cluster_ids": ", ".join(c["supporting_cluster_ids"]),
    "classified_tickets": int(n_by.get(c["category_id"], 0)), "pct_of_all_tickets": round(n_by.get(c["category_id"], 0) / len(df) * 100, 1),
    "revisit": "Y" if c.get("revisit") else "", "revisit_note": c.get("revisit_note", ""), "reviewer_notes": c.get("reviewer_notes", "")}
    for c in approved["categories"]]).sort_values("classified_tickets", ascending=False)

amap = {c["cluster_id"]: c["approved_category"] for c in approved["cluster_mapping"]}
card = {c["cluster_id"]: c for c in all_cards}
cl_sheet = disc.assign(
    unique_note_count=disc["cluster_id"].map(lambda k: card[k]["unique_notes"]),
    top_keywords=disc["cluster_id"].map(lambda k: ", ".join(card[k].get("keywords", [])[:8])),
    approved_category=disc["cluster_id"].map(amap),
    representative_notes=disc["cluster_id"].map(lambda k: "\n".join((card[k].get("representative_notes") or card[k].get("sample_notes", []))[:5])),
    supporting_examples=disc["supporting_examples"].map(lambda v: " | ".join(v)),
    contradictory_examples=disc["contradictory_examples"].map(lambda v: " | ".join(v)),
)[["cluster_id", "unique_note_count", "cluster_size", "top_keywords", "dominant_theme", "proposed_category", "approved_category",
   "cluster_type", "confidence", "recommended_action", "representative_notes", "supporting_examples", "contradictory_examples"]] \
   .rename(columns={"cluster_size": "original_record_count"})

# cluster vs per-note classification: how far each cluster's tickets agree with the category the cluster was mapped to
cv = df[df["processing_status"] == "VALID_TEXT"].assign(mapped=lambda x: x["cluster_id"].map(
        lambda c: amap.get("O-SINGLES" if c == -1 else f"C{int(c):02d}") if pd.notna(c) else None))
cv.loc[cv["cluster_id"] == -1, "mapped"] = None     # outliers were reviewed as mini-themes, not one cluster
agree = (cv.groupby("cluster_id").apply(lambda g: pd.Series({
            "tickets": len(g), "unique_notes": g["note_key"].nunique(),
            "mapped_category": g["mapped"].iloc[0],
            "pct_tickets_matching_mapped": round((g["primary_reason_id"] == g["mapped"]).mean() * 100, 1) if pd.notna(g["mapped"].iloc[0]) else None,
            "top_classified_category": g["primary_reason_id"].value_counts().index[0],
            "pct_top_category": round(g["primary_reason_id"].value_counts(normalize=True).iloc[0] * 100, 1),
            "categories_present": g["primary_reason_id"].nunique()}), include_groups=False)
         .reset_index())
agree["cluster_id"] = agree["cluster_id"].map(lambda c: "-1 (outliers)" if c == -1 else f"C{int(c):02d}")
agree = agree.merge(disc[["cluster_id", "cluster_type", "dominant_theme"]], on="cluster_id", how="left").sort_values("tickets", ascending=False)
cluster_x_cat = pd.crosstab(cv["cluster_id"].map(lambda c: "-1 (outliers)" if c == -1 else f"C{int(c):02d}"), cv["primary_reason_id"])
print("Cluster vs classification agreement (by ticket):")
display(agree.head(15))
w_ag = agree.dropna(subset=["pct_tickets_matching_mapped"])
print(f"Ticket-weighted share matching the cluster's mapped category: "
      f"{(w_ag['pct_tickets_matching_mapped'] * w_ag['tickets']).sum() / w_ag['tickets'].sum():.1f}% "
      f"| PURE clusters: {(lambda a: (a['pct_tickets_matching_mapped'] * a['tickets']).sum() / a['tickets'].sum())(w_ag[w_ag['cluster_type'] == 'PURE']):.1f}% "
      f"| MIXED/UNCERTAIN: {(lambda a: (a['pct_tickets_matching_mapped'] * a['tickets']).sum() / a['tickets'].sum())(w_ag[w_ag['cluster_type'] != 'PURE']):.1f}%")

revisit = pd.DataFrame([{"type": "category", "id": c["category_id"], "name / category": c["category_name"], "revisit_note": c.get("revisit_note", "")}
                        for c in approved["categories"] if c.get("revisit")] +
                       [{"type": "cluster", "id": c["cluster_id"], "name / category": c["approved_category"], "revisit_note": c.get("revisit_note", "")}
                        for c in approved["cluster_mapping"] if c.get("revisit")], columns=["type", "id", "name / category", "revisit_note"])
history = pd.DataFrame([{"taxonomy_version": (h := json.loads(p.read_text(encoding="utf-8")))["taxonomy_version"],
                         "approved_at": h["approved_at"], "categories": len(h["categories"]),
                         "redirected_clusters": sum(1 for c in h.get("cluster_mapping", []) if c.get("redirect_to")),
                         "current": "Y" if h["taxonomy_version"] == TAXV else ""} for p in sorted(HIST.glob("taxonomy_approved_*.json"))])

prim = df["primary_reason_id"].value_counts().rename_axis("primary_reason_id").reset_index(name="tickets")
prim["primary_reason_name"] = prim["primary_reason_id"].map(name); prim["main_category"] = prim["primary_reason_id"].map(MAIN_MAP); prim["pct"] = (prim["tickets"] / len(df) * 100).round(1)
sec = pd.Series([s for v in df["secondary_reason_ids"].fillna("") for s in split_ids(v)]).value_counts() \
        .rename_axis("secondary_reason_id").reset_index(name="tickets") if (df["secondary_reason_ids"].fillna("") != "").any() \
        else pd.DataFrame(columns=["secondary_reason_id", "tickets"])
ctype = df.loc[df["processing_status"] == "VALID_TEXT", "classification_type"].value_counts().rename_axis("classification_type").reset_index(name="tickets")
confd = conf_hist.rename_axis("confidence_band").reset_index(name="tickets"); confd["confidence_band"] = confd["confidence_band"].astype(str)
kpis = pd.DataFrame([("Total tickets", len(df)), ("Blank notes", int((df["primary_reason_id"] == "BLANK").sum())),
                     ("Classified (not BLANK / UNKNOWN)", int((~df["primary_reason_id"].isin(["BLANK", "UNKNOWN"])).sum())),
                     ("Unknown / unclassified", int((df["primary_reason_id"] == "UNKNOWN").sum())),
                     ("Approved categories", len(CAT_IDS)),
                     ("% flagged for review", round(df["review_required"].fillna(False).astype(bool).mean() * 100, 1)),
                     ("Taxonomy version", TAXV)], columns=["measure", "value"])

meta = pd.DataFrame([
    ("Run started", RUN["started"]), ("Workbook written", dt.datetime.now().isoformat(timespec="seconds")),
    ("Input file", CONFIG["input_path"]), ("Sheet", CONFIG["sheet_name"]), ("Note column", TEXT), ("ID column", CONFIG["id_col"]),
    ("Date filter", f"{CONFIG['date_col']} <= {CONFIG['date_max']}"), ("Total records processed", len(df)),
    ("Unique non-blank notes", len(unique_notes)), ("Embedding model", CONFIG["embedding_model"]),
    ("UMAP params", json.dumps(CONFIG["umap"])), ("HDBSCAN params", json.dumps(CONFIG["hdbscan"] | {"min_cluster_size": SEL})),
    ("Clustering configs compared", ", ".join(map(str, CONFIG["min_cluster_sizes"]))),
    ("LLM backend / model", f"{CONFIG['llm_backend']} / {MODEL}"),
    ("Prompt versions", f"taxonomy {CONFIG['taxonomy_prompt_version']}, classification {PV}"),
    ("Taxonomy version", TAXV), ("Taxonomy approved at", approved["approved_at"]),
    ("Confidence threshold", CONFIG["confidence_threshold"]),
    ("Processing status", json.dumps(RUN["status"])), ("Warnings", " | ".join(RUN["warnings"]) or "none"),
    ("Errors", " | ".join(RUN["errors"]) or "none"),
    ("Note on ground truth", "Only human-reviewed validation rows are treated as truth; structured form fields are unverified before Apr 2024 and are not used."),
], columns=["item", "value"])
print("Sheets prepared")

In [ ]:
WB = OUT / CONFIG["output_workbook"]
def add_table(ws, frame, r0=0, c0=0, name=None, widths=None, wrap_cols=(), fmt=None):
    cols = [{"header": str(c)} for c in frame.columns]
    data = frame.astype(object).where(frame.notna(), None).values.tolist()
    ws.add_table(r0, c0, r0 + max(len(frame), 1), c0 + len(frame.columns) - 1,
                 {"columns": cols, "data": data, "name": name, "style": "Table Style Light 9"})
    for i, c in enumerate(frame.columns):
        w = (widths or {}).get(c, min(max(len(str(c)) + 2, 12), 40))
        ws.set_column(c0 + i, c0 + i, w, fmt.get("wrap") if c in wrap_cols else None)

try:
    with pd.ExcelWriter(WB, engine="xlsxwriter", datetime_format="yyyy-mm-dd", engine_kwargs={"options": {"strings_to_urls": False, "default_date_format": "yyyy-mm-dd"}}) as xw:
        wb = xw.book
        fmt = {"wrap": wb.add_format({"text_wrap": True, "valign": "top"}), "h": wb.add_format({"bold": True, "font_size": 13}),
               "pct": wb.add_format({"num_format": "0.0"})}
        # 1 Classified Tickets
        t = xl_safe(tickets); ws = wb.add_worksheet("Classified Tickets"); xw.sheets["Classified Tickets"] = ws
        add_table(ws, t, name="ClassifiedTickets", widths={TEXT: 60, "note_clean": 50, "classification_evidence": 40, "review_reason": 30,
                                                          "primary_reason_name": 30, "secondary_reason_names": 30})
        ws.freeze_panes(1, 2)
        # 2 Approved Taxonomy
        ws = wb.add_worksheet("Approved Taxonomy")
        add_table(ws, xl_safe(tax_sheet), name="ApprovedTaxonomy", widths={"definition": 60, "inclusion_rules": 60, "exclusion_rules": 50,
                  "examples": 50, "subcategories": 45, "category_name": 32, "category_id": 30},
                  wrap_cols=("definition", "inclusion_rules", "exclusion_rules", "examples", "subcategories"), fmt=fmt)
        ws.freeze_panes(1, 1)
        # 3 Cluster Analysis
        ws = wb.add_worksheet("Cluster Analysis")
        add_table(ws, xl_safe(cl_sheet), name="ClusterAnalysis", widths={"dominant_theme": 45, "representative_notes": 70, "top_keywords": 35,
                  "supporting_examples": 50, "contradictory_examples": 50, "proposed_category": 28, "approved_category": 28},
                  wrap_cols=("dominant_theme", "representative_notes", "supporting_examples", "contradictory_examples"), fmt=fmt)
        ws.freeze_panes(1, 1)
        # 4 Summary Dashboard
        ws = wb.add_worksheet("Summary Dashboard")
        ws.write(0, 0, "UB disengagement reasons - summary", fmt["h"])
        add_table(ws, kpis, 2, 0, "KPIs", {"measure": 34, "value": 16})
        r = 2 + len(kpis) + 3
        ws.write(r - 1, 0, "Primary reason distribution", fmt["h"]); add_table(ws, prim, r, 0, "PrimaryDist", {"primary_reason_id": 32, "primary_reason_name": 40})
        r2 = r + len(prim) + 3
        ws.write(r2 - 1, 0, "Secondary reason frequency", fmt["h"]); add_table(ws, sec if len(sec) else pd.DataFrame({"secondary_reason_id": ["(none)"], "tickets": [0]}), r2, 0, "SecondaryFreq")
        r3 = r2 + max(len(sec), 1) + 3
        ws.write(r3 - 1, 0, "Explicit vs inferred (non-blank notes)", fmt["h"]); add_table(ws, ctype, r3, 0, "ClassType")
        r4 = r3 + len(ctype) + 3
        ws.write(r4 - 1, 0, "Confidence distribution (non-blank notes)", fmt["h"]); add_table(ws, confd, r4, 0, "ConfDist")
        r5 = r4 + len(confd) + 3
        ws.write(r5 - 1, 0, "Top 10 most common raw notes", fmt["h"]); add_table(ws, xl_safe(top_raw_notes.head(10)), r5, 0, "TopRawNotes", {"raw_note": 70})
        r6 = r5 + 13
        ws.write(r6 - 1, 0, "Operational metrics", fmt["h"]); add_table(ws, ops, r6, 0, "OpsMetrics")
        def bar(title, table_row, n, cat_col, val_col, anchor, horiz=True, h=None):
            ch = wb.add_chart({"type": "bar" if horiz else "column"})
            ch.add_series({"categories": ["Summary Dashboard", table_row + 1, cat_col, table_row + n, cat_col],
                           "values": ["Summary Dashboard", table_row + 1, val_col, table_row + n, val_col],
                           "fill": {"color": "#3B6FB6"}, "gap": 60, "data_labels": {"value": True, "font": {"color": "#555555", "size": 8}}})
            ch.set_title({"name": title, "name_font": {"size": 11, "bold": False}}); ch.set_legend({"none": True})
            ch.set_y_axis({"reverse": True} if horiz else {"major_gridlines": {"visible": True, "line": {"color": "#E5E5E5"}}})
            ch.set_x_axis({"major_gridlines": {"visible": True, "line": {"color": "#E5E5E5"}}} if horiz else {})
            ch.set_size({"width": 620, "height": h or max(260, 22 * n + 80)}); ws.insert_chart(anchor, ch)
        bar("Primary reason (tickets)", r, len(prim), 0, 1, "G3")
        if len(sec): bar("Secondary reasons (tickets)", r2, len(sec), 0, 1, "G33")
        bar("Classification type", r3, len(ctype), 0, 1, "Q3", horiz=False, h=260)
        bar("Confidence (non-blank notes)", r4, len(confd), 0, 1, "Q18", horiz=False, h=280)
        ws.set_column(0, 0, 40); ws.set_column(1, 3, 16)
        # 5 Validation (mirror)
        ws = wb.add_worksheet("Validation")
        ws.write(0, 0, f"Mirror of {VAL_XLSX.name} - reviewers edit that file; this sheet is refreshed each run.", fmt["h"])
        if len(val):
            add_table(ws, xl_safe(val.drop(columns=["note_clean"], errors="ignore")), 2, 0, "ValidationSample",
                      {"note_original": 60, "predicted_evidence": 35, "reviewer_comments": 35}, wrap_cols=("note_original",), fmt=fmt)
        r = 2 + len(val) + 3
        if vm:
            ws.write(r, 0, "Validation metrics (reviewed rows only)", fmt["h"]); add_table(ws, vm["summary"], r + 1, 0, "ValMetrics")
            add_table(ws, vm["per_category"], r + len(vm["summary"]) + 4, 0, "ValPerCategory")
            cm = vm["confusion"].reset_index().rename(columns={"index": "human \\ predicted"})
            add_table(ws, cm, r + len(vm["summary"]) + len(vm["per_category"]) + 7, 0, "ValConfusion")
        # Cluster vs Category
        ws = wb.add_worksheet("Cluster vs Category")
        ws.write(0, 0, "How each cluster's tickets were classified note by note (clusters were a discovery aid, not labels)", fmt["h"])
        add_table(ws, xl_safe(agree), 2, 0, "ClusterAgreement", {"dominant_theme": 50, "mapped_category": 28, "top_classified_category": 28})
        add_table(ws, cluster_x_cat.reset_index().rename(columns={"cluster_id": "cluster"}), len(agree) + 6, 0, "ClusterXCategory")
        # 6 Revisit list, 7 Taxonomy history
        ws = wb.add_worksheet("Revisit list")
        add_table(ws, revisit if len(revisit) else pd.DataFrame({"type": ["(nothing flagged)"], "id": [""], "name / category": [""], "revisit_note": [""]}),
                  name="RevisitList", widths={"revisit_note": 70, "name / category": 35})
        ws = wb.add_worksheet("Taxonomy history")
        add_table(ws, history, name="TaxHistory", widths={"taxonomy_version": 20, "approved_at": 22})
        if changelog is not None:
            ws.write(len(history) + 3, 0, "Change log vs previous approved version", fmt["h"])
            add_table(ws, xl_safe(changelog), len(history) + 4, 0, "ChangeLog", {"change": 30, "item": 32, "detail": 90}, wrap_cols=("detail",), fmt=fmt)
        # 8 Profiling, 9 Run Metadata
        ws = wb.add_worksheet("Profiling")
        add_table(ws, profile, 0, 0, "Profile", {"metric": 42})
        add_table(ws, xl_safe(top_raw_notes), len(profile) + 3, 0, "TopRawNotesAll", {"raw_note": 80})
        add_table(ws, kind_year.reset_index().rename(columns={"Year Disengagement": "year"}), len(profile) + len(top_raw_notes) + 6, 0, "NoteKindByYear")
        ws = wb.add_worksheet("Run Metadata")
        add_table(ws, xl_safe(meta), name="RunMetadata", widths={"item": 30, "value": 120}, wrap_cols=("value",), fmt=fmt)
    logger.info(f"Workbook written: {WB}")
    print(f"Workbook written: {WB}")
except Exception as e:
    RUN["errors"].append(f"Excel export failed: {e}"); logger.error(f"Excel export failed: {e}"); raise
RUN["status"]["phase9"] = "workbook written"

## Phase 10 · Categorized copy of the source workbook

This step writes **`<input> - categorized.xlsx`** next to the input file. It is a copy of the whole source workbook with `main_cat`, `sub_cat` and `detail_cat` inserted straight after `Ticket no.` (columns L:N). The columns are filled for the in-scope rows (to `date_max`); later rows stay blank.

* Excel itself does the work (COM). Inserting columns that way shifts every formula reference; openpyxl would break them.
* The copy is **values only**. Every sheet is pasted as values with PasteSpecial, so text stays text and errors stay errors, and external links are removed. The original file is never touched, and reading it works while it's open in Excel.
* Filters are cleared first. The step refuses to write if any rows are hidden, because a bulk write into a filtered sheet lands in the wrong rows.
* The copy is built in `artifacts/` and checked cell by cell before it replaces the old copy. The checks cover the three new columns against the tickets and every original column against the source. The previous copy is kept as `... - categorized - previous.xlsx`.
* If the old copy is open in Excel it can't be replaced. The new one stays in `artifacts/` and a warning says so.


In [ ]:
# ---- Phase 10. Categorized copy of the source workbook (values only) -----------------
import shutil
CAT_COLS = {"main_cat": "main_category", "sub_cat": "primary_reason_id", "detail_cat": "detail_category"}

def _norm(s):
    def f(x):
        if x is None or x is pd.NaT or (isinstance(x, float) and np.isnan(x)): return ""
        if isinstance(x, str):
            x = x.replace("\r\n", "\n").strip(); return "" if x == "0" else x
        return "" if (not isinstance(x, bool) and x == 0) else str(x)
    return s.map(f)

def build_categorized_copy():
    src = Path(CONFIG["input_path"])
    dst = src.with_name(f"{src.stem} - categorized{src.suffix}")
    prev = src.with_name(f"{src.stem} - categorized - previous{src.suffix}")
    tmp = ART / f"_categorized_build{src.suffix}"
    shutil.copy2(src, tmp)
    raw = pd.read_excel(tmp, sheet_name=CONFIG["sheet_name"])              # cached formula results, for the check afterwards
    vals = {int(r): tuple(df.at[i, c] if pd.notna(df.at[i, c]) else None for c in CAT_COLS.values())
            for i, r in df["source_row"].items()}
    import pythoncom, win32com.client as w32
    pythoncom.CoInitialize()
    xl = w32.DispatchEx("Excel.Application")                                # own instance: the user's open Excel is left alone
    xl.Visible = False; xl.DisplayAlerts = False; xl.AskToUpdateLinks = False
    links = None
    try:
        wb = xl.Workbooks.Open(str(tmp), 0, False)                         # UpdateLinks=0: keep cached values of external links
        xl.Calculation = -4135                                              # manual: results stay as last saved
        ws = wb.Worksheets(CONFIG["sheet_name"])
        if ws.FilterMode: ws.ShowAllData()
        last = ws.UsedRange.Row + ws.UsedRange.Rows.Count - 1
        if ws.Range(f"A1:A{last}").SpecialCells(12).Count != last:          # 12 = visible cells
            raise RuntimeError("Rows are hidden in the source sheet; not writing (a bulk write would land in the wrong rows)")
        hdr = [ws.Cells(1, j).Value for j in range(1, ws.UsedRange.Columns.Count + 1)]
        kcol = hdr.index(CONFIG["id_col"]) + 1
        if hdr[kcol:kcol + 3] == list(CAT_COLS):
            raise RuntimeError("The source already has main_cat / sub_cat / detail_cat columns")
        ws.Range(ws.Cells(1, kcol + 1), ws.Cells(1, kcol + 3)).EntireColumn.Insert()
        data = [tuple(CAT_COLS)] + [vals.get(r, (None, None, None)) for r in range(2, last + 1)]
        ws.Range(ws.Cells(1, kcol + 1), ws.Cells(last, kcol + 3)).Value = data
        ws.Range(ws.Cells(1, kcol + 1), ws.Cells(1, kcol + 3)).Font.Bold = True
        ws.Range(ws.Cells(1, kcol + 1), ws.Cells(1, kcol + 3)).EntireColumn.ColumnWidth = 26
        for sh in wb.Worksheets:                                            # values only: exact values, text stays text
            ur = sh.UsedRange
            try: nform = ur.SpecialCells(-4123).Count
            except Exception: nform = 0
            if nform:
                sh.Activate(); ur.Copy(); ur.PasteSpecial(-4163); xl.CutCopyMode = False; sh.Range("A1").Select()
        links = wb.LinkSources(1)
        for l in (links or []): wb.BreakLink(l, 1)
        xl.Calculation = -4105
        wb.Worksheets(1).Activate(); wb.Save(); wb.Close(False)
    finally:
        xl.Quit(); del xl; pythoncom.CoUninitialize()

    # check cell by cell before replacing anything
    out = pd.read_excel(tmp, sheet_name=CONFIG["sheet_name"])
    assert list(out.columns[kcol:kcol + 3]) == list(CAT_COLS), out.columns[kcol - 1:kcol + 4].tolist()
    assert len(out) == len(raw), (len(out), len(raw))
    orig = [c for c in out.columns if c not in CAT_COLS]
    bad = {c: int((_norm(out[c]) != _norm(raw[c])).sum()) for c in orig}
    bad = {c: n for c, n in bad.items() if n}
    assert not bad, f"Original columns changed in the copy: {bad}"
    rows = df["source_row"].to_numpy() - 2
    rest = np.setdiff1d(np.arange(len(out)), rows)
    for k, c in CAT_COLS.items():
        assert (out[k].iloc[rows].fillna("").astype(str).to_numpy() == df[c].fillna("").astype(str).to_numpy()).all(), f"{k} mismatch"
        assert out[k].iloc[rest].isna().all(), f"{k} filled outside scope"
    try:
        if dst.exists(): shutil.copy2(dst, prev)
        shutil.copy2(tmp, dst)
    except PermissionError:
        warn(f"{dst.name} is open in Excel - the new copy is at {tmp}; close it and rerun this cell")
        return tmp
    print(f"Categorized copy: {dst} | {len(out):,} rows, {len(df):,} filled, external links removed: {len(links or [])}, "
          f"all {len(orig)} original columns match the source")
    return dst

if CONFIG.get("categorized_copy") and CONFIG["input_type"] == "excel" and os.name == "nt":
    CATEGORIZED = build_categorized_copy()
    RUN["status"]["phase10"] = f"categorized copy: {CATEGORIZED.name}"


## Phase 11 · Backfill the form structure (tickets to Mar-2024)

This step writes **`<input> - backfill to <date_max>.xlsx`** next to the input file, one row per in-scope ticket. Its columns follow the sheet's own:
* `Year Disengagement`, `Month Disengagement`, `Date Disengagement` and `Ticket no.` come from the source.
* **`Vol/Inv_backfilled`** comes from `main_cat`: Resignation → Voluntary; Service Dissatisfaction / Business Reason → Involuntary; For Record → For Record Purpose Only; Deleted → Deleted; no reason → `-`.
* **`UB Type_backfill`** is the form-tree value that matches the label, written the way the sheet's `UB Type` writes it. The label wins, as everywhere in this notebook: the note first, then the form fields only where the note gives no reason. The ticket's own form value is used when it gives the same detail category. Otherwise the value comes from `BACKFILL_LEAF` (detail_cat → tree value). `UB Type_backfill_basis` says which applied:
  * `form (...)` means the value came from the ticket's own form;
  * `exact` means it's the only tree value for that detail;
  * `nearest` means it's a judgement call;
  * `branch only` means the reason has no sub-reason level.

  For Record, Deleted and no-reason tickets get `-`, as in the sheet.
* **`Master Reason_backfill`** = `main_cat`, except For Record is written *For Record Purpose Only*, as in the sheet's Master Reason. `main_cat`, `sub_cat` and `detail_cat` follow it.
* After those come the label's source and evidence, the note, and the form fields step 7b reads. The sheet's own `UB Type`, `Vol/Inv` and `Master Reason` are included for comparison.

A **Category definitions** sheet lists every main_cat → sub_cat → detail_cat with its definition, the default `UB Type_backfill` and the ticket count. The detail definitions are in `DETAIL_DEFINITIONS` in the cell below.

The step also checks the backfill. It reports any ticket where the derived Vol/Inv differs from the direction the form tree gives the backfilled value, and any where the backfilled value maps back to a different main category.


In [ ]:
# ---- Phase 11. Backfill the form structure for tickets to Mar-2024 --------------------
# UB Type_backfill = the form-tree value (as the sheet's 'UB Type' writes it) that matches our label. The label wins (note first,
# form fields only where the note gives no reason). Master Reason_backfill = main_cat; Vol/Inv_backfilled is derived from it.
VOL_LEAVES = set(VOLUNTARY)
LEAF_PREFIX = {"Type of Performance Issues": "Performance Issues", "Type of Attendance Issues": "Attendance Issues",
               "Type of Role Being Moved": "Role being moved"}
UBT_PREFER = {"Role no longer needed": "Type of End of Role"}        # leaf in two columns: the tree's main home

def leaf_label(v):
    """UB Type text -> (category_id, subcategory, detail_category) via the step-7b maps; None if not a tree value."""
    v = re.sub(r"\s+", " ", str(v)).strip()
    if not v or v == "-": return None
    def out(e): return (e[0], e[1], e[2] if len(e) > 2 else detail_of(e[0], e[1]))
    if v in VOLUNTARY: return out(VOLUNTARY[v])
    lo = v.lower()
    if lo in BRANCHES: return out(BRANCHES[lo][1])
    if lo in ("performance issues", "attendance issues"):
        return out(COLUMNS["Type of Service Dissatisfaction"]["Performance Issues" if lo == "performance issues" else "Attendance Issues"][2])
    if lo == "role being moved": return out(COLUMNS["Type of Business Reasons"]["Role being moved"][2])
    pre, sep, leaf = v.partition(" - ")
    for col, p in LEAF_PREFIX.items():
        if sep and pre.lower() == p.lower() and leaf in COLUMNS[col]: return out(COLUMNS[col][leaf])
    hits = [c for c, m in COLUMNS.items() if v in m and m[v][0] != "->"]
    if not hits: return None
    return out(COLUMNS[UBT_PREFER.get(v, hits[0]) if UBT_PREFER.get(v) in hits else hits[0]][v])

def path_leaf(evidence):
    """The tree value step 7b landed on, written the way 'UB Type' writes it."""
    seg = str(evidence).split("] ", 1)[-1].replace(" (neighbouring column)", "").split(" > ")[-1]
    col, sep, val = seg.partition(": ")
    if not sep: return seg.strip()
    for c, p in LEAF_PREFIX.items():
        if col.strip() == c.replace("Type of ", ""): return f"{p} - {val.strip()}"
    return val.strip()

# Our label -> nearest tree value when nothing better is known. 'exact' = the only tree value for that detail;
# 'nearest' = a judgement (several tree values fit, or none does). For Record has no further columns in the tree: '-'.
BACKFILL_LEAF = {
    "downturn_budget_cost_cutting": ("Budget Issues", "exact"), "downturn_downsizing": ("Downsizing", "exact"),
    "downturn_slowdown_volume": ("Business Slow Down", "exact"), "downturn_general": ("Business Slow Down", "nearest"), "downturn_covid": ("Business Slow Down", "nearest"),
    "downturn_seasonal": ("Business Slow Down", "nearest"),
    "business_reason_general": ("Business Reasons", "branch only"), "business_requirements_changed": ("Business Reasons", "nearest"),
    "business_client_decided_not_to_continue": ("Business Reasons", "nearest"), "business_structural_changes": ("Business Reasons", "nearest"),
    "account_pull_out": ("Client Contract Cancellation (All Staff Disengaged) - Client Pull-out", "exact"),
    "account_exit_general": ("Client Contract Cancellation (All Staff Disengaged) - Client Pull-out", "nearest"),
    "account_breach_of_contract": ("Client Contract Cancellation (All Staff Disengaged) - Client Pull-out", "nearest"),
    "account_contract_terminated": ("Client Contract Cancellation (All Staff Disengaged) - Client Pull-out", "nearest"),
    "account_sub_client_pull_out": ("Client Contract Cancellation (All Staff Disengaged) - Client Pull-out", "nearest"),
    "client_closure": ("Client Contract Cancellation (All Staff Disengaged) - Client Pull-out", "nearest"),
    "client_closure_general": ("Client Contract Cancellation (All Staff Disengaged) - Client Pull-out", "nearest"),
    "client_sold_acquired": ("Role being moved - Company Acquisition", "exact"),
    "buyout": ("Formal Staff Buy Out", "exact"), "buyout_direct_hire": ("Client wants to hire them direct (Customer Black Flag)", "exact"),
    "non_payment": ("Defaulted Client - Suspension for non payment - Bad Debt", "exact"),
    "role_moved_onshore": ("Role being moved - Moving role onshore", "exact"), "role_moved_provider": ("Role being moved - Moving Role to Another BPO", "exact"),
    "role_moved_in_house": ("Role being moved - Moving role onshore", "nearest"), "role_moved_general": ("Role being moved", "branch only"),
    "role_eliminated": ("Role no longer needed", "nearest"), "role_no_longer_needed_general": ("Role no longer needed", "nearest"), "role_automation": ("Role no longer needed", "nearest"),
    "role_project_ended": ("Contract ended no more work", "nearest"), "role_restructure": ("Company Restructure", "nearest"),
    "resignation_better_opportunity": ("Better opportunity for advancement", "nearest"), "resignation_career_studies": ("Continue Schooling/Study", "exact"),
    "resignation_dissatisfied": ("Dissatisfied with client training", "nearest"), "resignation_family": ("Family caring responsibilities", "exact"),
    "resignation_general": ("Personal Reasons Undisclosed", "nearest"), "resignation_personal_reasons": ("Personal Reasons Undisclosed", "exact"),
    "resignation_health": ("Medical - Physical Health", "nearest"), "resignation_own_business": ("Family business", "exact"),
    "resignation_relocation": ("Staff Moving Overseas", "nearest"), "resignation_schedule": ("Candidate - Shift Not Suitable", "nearest"),
    "resignation_work_arrangement": ("RTO - Personal Reasons", "nearest"),
    "attendance_absenteeism_tardiness": ("Attendance Issues - Unexplained/ Unauthorised absence", "nearest"),
    "attendance_ncns_awol": ("Attendance Issues - AWOL", "nearest"), "attendance_failed_rtwo": ("Return to Work Order Ignored", "exact"),
    "attendance_no_show_onboarding": ("Did not report for NHI", "nearest"), "attendance_general": ("Attendance Issues", "branch only"),
    "attendance_connectivity": ("Attendance Issues", "nearest"),
    "misconduct_code_of_conduct": ("Violation of client's code of conduct", "nearest"), "misconduct_fraud_falsification": ("Time Manipulation", "nearest"),
    "misconduct_call_avoidance": ("Violation of client's code of conduct", "nearest"), "misconduct_general": ("Violation of client's code of conduct", "nearest"),
    "performance_quality": ("Performance Issues - Unable to meet KPIs", "nearest"), "performance_failed_review": ("Performance Issues - Failed Performance Review", "exact"),
    "performance_failed_pip": ("Performance Issues - Failed PIP", "exact"), "performance_failed_training": ("Performance Issues - Training Difficulties", "nearest"),
    "performance_skills_role_fit": ("Performance Issues - Staff Mismatch", "nearest"),
    "performance_team_conflict": ("Personnel - Disengaged due to issues with other team members", "exact"),
    "performance_general": ("Service Dissatisfaction", "branch only"), "performance_communication": ("Performance Issues", "nearest"),
    "performance_undocumented": ("Performance Issues", "nearest"), "cloudstaff_service_cost_value": ("Service Dissatisfaction", "nearest"),
}
NO_FURTHER = {"FOR_RECORD_PURPOSE_ONLY": "For Record: no further columns", "FORM_DIRECTION_ONLY": "no reason given", "BLANK": "no reason given", "UNKNOWN": "no reason given",
              "DELETED": "Deleted: no further columns"}
MAIN_TO_VI = {"Resignation": "Voluntary", "Service Dissatisfaction": "Involuntary", "Business Reason": "Involuntary",
              "For Record": "For Record Purpose Only"}

def backfill_row(r):
    cid, det = r["primary_reason_id"], r["detail_category"]
    want = (cid, det)
    cands = [("form (UB Type agrees)", r.get("UB Type"))]
    if r["classification_source"] == "form fields":
        cands.append(("form (path)", path_leaf(r["classification_evidence"])))
    for basis, v in cands:
        lab = leaf_label(v) if isinstance(v, str) else None
        if lab and (lab[0], lab[2]) == want:
            return re.sub(r"\s+", " ", v).strip(), basis
    if cid in NO_FURTHER:
        return "-", NO_FURTHER[cid]
    if cid == "STAFF_MEDICAL" and r["primary_subcategory"] == "Deceased":
        return "Death - Staff member passed away", "exact"
    if det in BACKFILL_LEAF:
        return BACKFILL_LEAF[det]
    return "-", "NOT MAPPED"

bf = df.apply(backfill_row, axis=1, result_type="expand"); bf.columns = ["ubt_bf", "ubt_basis"]
df["UB Type_backfill"], df["UB Type_backfill_basis"] = bf["ubt_bf"], bf["ubt_basis"]
df["Master Reason_backfill"] = df["main_category"].replace({"For Record": "For Record Purpose Only"})   # the sheet's own Master Reason wording
df["Vol/Inv_backfilled"] = df["main_category"].map(MAIN_TO_VI)
df.loc[df["primary_reason_id"] == "DELETED", "Vol/Inv_backfilled"] = "Deleted"
df["Vol/Inv_backfilled"] = df["Vol/Inv_backfilled"].fillna("-")

# cross-check: the direction the form tree gives the backfilled value (Voluntary leaves are the AR list)
def tree_direction(v):
    if v == "-": return None
    return "Voluntary" if v in VOL_LEAVES else "Involuntary"
tdir = df["UB Type_backfill"].map(tree_direction)
vi_diff = tdir.notna() & df["Vol/Inv_backfilled"].isin(["Voluntary", "Involuntary"]) & (tdir != df["Vol/Inv_backfilled"])
round_trip = df["UB Type_backfill"].map(leaf_label)
rt_main = round_trip.map(lambda e: MAIN_MAP.get(e[0]) if e else None)
mismatch_main = round_trip.notna() & (rt_main != df["main_category"])
unmapped = int((df["UB Type_backfill_basis"] == "NOT MAPPED").sum())
print(f"Backfilled {len(df):,} tickets | basis:"); display(df["UB Type_backfill_basis"].value_counts().rename_axis("basis").reset_index(name="tickets"))
print(f"Vol/Inv from main_cat differs from the tree's direction for {int(vi_diff.sum())} tickets; "
      f"backfilled UB Type maps back to another main_cat for {int(mismatch_main.sum())}; not mapped: {unmapped}")
if vi_diff.any():
    display(df.loc[vi_diff].groupby(["primary_reason_id", "UB Type_backfill", "Vol/Inv_backfilled"]).size().rename("tickets").reset_index())

DETAIL_DEFINITIONS = {   # detail_cat definitions (written from the taxonomy subcategories and rules, 2026-11-25)
    'resignation_general': "Staff resigned and the note gives no reason (e.g. 'Staff resigned', 'Resignation').",
    'resignation_personal_reasons': "Staff resigned citing 'personal reasons' with no further detail.",
    'resignation_better_opportunity': 'Staff left for another job, a better offer or higher pay, including no salary increase, an employer counter-offer or a failed salary negotiation (also for candidates before their start).',
    'resignation_career_studies': 'Staff left for career growth, a career change or to study.',
    'resignation_health': 'Staff left because of their own health: resigned for health or medical reasons (physical, mental, fear of COVID), or separated as unfit to work, did not start due to illness, failed the medical requirements, on prolonged sick leave, unable to work night shift for health, or deceased. Includes every STAFF_MEDICAL ticket.',
    'resignation_family': 'Staff left for family or caring responsibilities (children, an ill relative, family matters).',
    'resignation_relocation': 'Staff left because they are moving to another city or overseas.',
    'resignation_work_arrangement': 'Staff left over the work arrangement: cannot or will not return to the office, wants permanent work from home, commute.',
    'resignation_own_business': 'Staff left to start or run their own or the family business, or to freelance.',
    'resignation_schedule': 'Staff left because of the schedule or shift (e.g. night shift, hours) with no health reason given, including a refusal to work the graveyard shift (recorded on the form as Cloudstaff-initiated).',
    'resignation_dissatisfied': 'Staff left because they were unhappy with Cloudstaff, the client (incl. client training or mistreatment), their manager or team leader, or the team.',
    'performance_general': 'Client released the staff for performance or service reasons, with no more specific detail.',
    'performance_quality': "Work quality, productivity, efficiency, errors, KPIs or expectations not met; also 'Performance Issues' on the form with no sub-reason.",
    'performance_undocumented': 'Performance is the reason, but the ticket says it is not sufficiently documented by the client.',
    'performance_skills_role_fit': 'Staff lacks the skills or experience, or is not a fit for the role.',
    'performance_failed_training': 'Staff failed training, onboarding or probation, or was not regularised.',
    'performance_failed_pip': 'Staff failed or declined a performance improvement plan (PIP).',
    'performance_communication': 'Communication problems: language, responsiveness, keeping the client updated.',
    'cloudstaff_service_cost_value': "Client is unhappy with Cloudstaff's own service, price or value, rather than with the staff member.",
    'attendance_general': 'Attendance is the reason, with no more specific detail.',
    'attendance_absenteeism_tardiness': 'Excessive or unexplained absences, lateness, early logouts or undertime, including transport issues.',
    'attendance_ncns_awol': 'No call no show, AWOL, job abandonment or the staff became unreachable.',
    'attendance_failed_rtwo': 'Staff did not comply with a return-to-work order (RTWO).',
    'attendance_no_show_onboarding': 'Staff did not turn up on day one or after onboarding (NHI), or the start never happened.',
    'attendance_connectivity': 'Staff could not work because of internet or power outages.',
    'misconduct_general': 'A conduct or integrity breach, with no more specific detail.',
    'misconduct_call_avoidance': 'Avoiding calls or work while logged in.',
    'misconduct_fraud_falsification': 'Time or stats manipulation, fraud, falsified documents or medical certificates, or a PCI (card data) violation.',
    'misconduct_code_of_conduct': 'Code-of-conduct violation: unprofessional, rude or hostile behaviour, a malicious act, breach of confidentiality, or dismissal after an admin hearing.',
    'business_reason_general': "'Business reason' or 'business decision' with no specific driver.",
    'business_client_decided_not_to_continue': 'Client decided not to proceed, continue or keep the engagement, with no staff-fault reason (e.g. a deal retracted, or the client treated it as a 90-day trial).',
    'business_requirements_changed': "The client's requirements, plans or approach changed.",
    'business_structural_changes': "Generic business and structural changes with no role elimination stated (includes the old 'Business & Structural changes' dropdown value seen in notes).",
    'downturn_general': 'Business downturn, with no more specific detail.',
    'downturn_slowdown_volume': 'Reduced workload, volume, sales or customers.',
    'downturn_covid': "Impact of COVID or lockdowns on the client's business.",
    'downturn_seasonal': 'Seasonal or off-peak lull.',
    'downturn_budget_cost_cutting': 'Budget cuts, cost reduction or the client can no longer afford the role.',
    'role_no_longer_needed_general': 'The client no longer needs the role, with no more specific detail.',
    'role_eliminated': 'The role is redundant, eliminated or closed.',
    'role_restructure': 'A restructure, reorganisation or change of direction removes the role.',
    'role_automation': 'Automation or a process change removes the role.',
    'role_project_ended': 'A project, temporary or fixed-term role came to an end.',
    'role_moved_general': 'The work moved away from Cloudstaff, with no more specific detail.',
    'role_moved_onshore': 'The role moved onshore or to a local hire.',
    'role_moved_in_house': "The work was taken into the client's in-house team or existing staff.",
    'role_moved_provider': 'The work moved to another (often cheaper) provider, competitor or offshore location.',
    'client_closure_general': 'The client is closing or was acquired, with no more specific detail.',
    'client_closure': 'The client, sub-client or team is closing, dissolving or ceasing trading.',
    'client_sold_acquired': 'The client was sold, acquired or merged, ending the engagement.',
    'account_exit_general': 'The client ends its whole account with Cloudstaff, with no more specific detail.',
    'account_contract_terminated': 'The client terminated its contract or agreement with Cloudstaff.',
    'account_pull_out': 'The client pulled out its account or staff.',
    'account_sub_client_pull_out': 'A sub-client of the client pulled out or did not renew.',
    'account_breach_of_contract': 'The account ended because of a breach of contract.',
    'non_payment': 'Cloudstaff suspended, deactivated or terminated the account because the client did not pay (incl. a client that went unresponsive to payment).',
    'buyout': "The client bought out the staff's contract and continues with them outside Cloudstaff.",
    'buyout_direct_hire': 'The client hires the staff directly.',
    'record_general': 'Administrative record, with no more specific detail.',
    'record_form_value': "The note or the form's direction dropdown says 'For Record Purpose Only' (or 'for documentation only') with no further detail.",
    'record_work_arrangement_change': 'Re-papering for a work-arrangement change (OfficeFlex, Staff on Fiber, permanent WFH).',
    'record_entity_billing_change': 'Re-papering for a client entity, billing, currency, MSA or account change.',
    'record_engagement_type_change': 'Re-papering for an engagement-type change (e.g. to full time, CS Now to full time).',
    'record_internal_transfer': 'Re-papering for an internal or lateral transfer within Cloudstaff, including an orchestrated move between clients or a move between MSAs.',
    'form_direction_general': 'The note is only a form direction value.',
    'form_voluntary': "The note is only 'Voluntary' copied from the form: staff-initiated, reason not given.",
    'form_involuntary': "The note is only 'Involuntary' copied from the form: client- or Cloudstaff-initiated, reason not given.",
    'deleted': 'The ticket was deleted or voided; the staff is not actually leaving.',
    'deleted_cancelled_withdrawn': 'The disengagement was cancelled or withdrawn; the staff stays.',
    'blank': 'The note is empty, a placeholder or meaningless text.',
    'unknown': "The note has text, but no reason that fits a category (e.g. 'Other', dates or notice periods only).",
    'performance_failed_review': "Staff failed a formal performance review. Set from the form ('Failed Performance Review'); tickets categorised from the note keep performance_quality.",
    'performance_team_conflict': "Released because of issues with other team members. Set from the form ('Personnel - Disengaged due to issues with other team members'); note-based tickets keep performance_general.",
    'downturn_downsizing': "Client reduced headcount (downsizing). Set from the form ('Downsizing'); note-based tickets keep downturn_budget_cost_cutting.",
}
MAIN_DEFINITIONS = {
    "Resignation": "The staff member chose to leave, or left because of their own health.",
    "Service Dissatisfaction": "The client (or Cloudstaff) ended the engagement because of the staff member: performance, attendance or conduct. Also holds the few cases where the client is unhappy with Cloudstaff's own service, price or value.",
    "Business Reason": "The client's business decision, not caused by the staff member: downturn, role no longer needed or moved, closure, account exit, buy-out, non-payment.",
    "For Record": "Administrative re-papering; the staff member is not leaving Cloudstaff (work-arrangement, entity or engagement-type changes, internal transfers).",
    "Other/Unknown": "No reason can be read: blank note, deleted ticket, or text with no reason.",
}
SUB_CAT_NOTES = {   # appended to the approved definition in the Category definitions sheet
    "STAFF_MEDICAL": "From the form: Cloudstaff-initiated 'Medical Issues' (did not start due to illness, failed medical requirements, death).",
    "NON_PAYMENT": "From the form: 'Defaulted Client - Suspension for non payment - Bad Debt', or 'client has gone unresponsive' in UB Main Reason.",
    "FORM_DIRECTION_ONLY": "Interim label only: these tickets are re-read from the form fields (step 7b), so none keep it.",
    "BLANK": "Kept only when the form fields give no reason either.",
    "UNKNOWN": "Kept only when the form fields give no reason either.",
}
MAIN_ORDER = ["Resignation", "Service Dissatisfaction", "Business Reason", "For Record", "Other/Unknown"]
SUB_ORDER = ["RESIGNATION", "STAFF_MEDICAL", "SERVICE_DISSATISFACTION", "ATTENDANCE_AWOL", "MISCONDUCT", "BUSINESS_REASON",
             "BUSINESS_DOWNTURN", "ROLE_NO_LONGER_NEEDED", "ROLE_MOVED_ONSHORE_OR_PROVIDER", "CLIENT_CLOSURE_OR_ACQUISITION",
             "CLIENT_ACCOUNT_EXIT", "NON_PAYMENT", "CLIENT_BUYOUT", "FOR_RECORD_PURPOSE_ONLY", "FORM_DIRECTION_ONLY", "DELETED", "BLANK", "UNKNOWN"]
FORM_ONLY_DETAIL = {"SERVICE_DISSATISFACTION": ["performance_failed_review", "performance_team_conflict"], "BUSINESS_DOWNTURN": ["downturn_downsizing"]}

def category_definitions(tickets):
    """One row per detail_cat: main / sub / detail with definitions, the default UB Type_backfill and the tickets in `tickets`."""
    n = tickets.groupby(["sub_cat", "detail_cat"]).size()
    rows = []
    for m in MAIN_ORDER:
        for sc in [x for x in SUB_ORDER if MAIN_MAP.get(x) == m] + sorted(x for x in CAT_IDS if MAIN_MAP.get(x) == m and x not in SUB_ORDER):
            seen = list(dict.fromkeys(list(DETAIL_MAP.get(sc, {}).values()) + FORM_ONLY_DETAIL.get(sc, [])))
            seen += sorted(d for (s_, d) in n.index if s_ == sc and d not in seen)          # anything used but not in the maps
            for d in seen:
                leaf, basis = ("-", NO_FURTHER[sc]) if sc in NO_FURTHER else BACKFILL_LEAF.get(d, ("", ""))
                rows.append((m, MAIN_DEFINITIONS.get(m, ""), sc, (CAT[sc]["definition"] + (" " + SUB_CAT_NOTES[sc] if sc in SUB_CAT_NOTES else "")) if sc in CAT else "", d,
                             DETAIL_DEFINITIONS.get(d, ""), leaf, basis, int(n.get((sc, d), 0))))
    out_ = pd.DataFrame(rows, columns=["main_cat", "main_cat definition", "sub_cat", "sub_cat definition", "detail_cat",
                                       "detail_cat definition", "default UB Type_backfill", "basis", "tickets in this file"])
    missing = sorted(set(out_.loc[out_["detail_cat definition"] == "", "detail_cat"]))
    if missing: warn(f"detail_cat without a definition: {missing}")
    return out_

FIELD_COLS = [TEXT, "Reason for Disengagement", "Type of Voluntary Reason", "Type of Involuntary", "Type of Customer Initiated",
              "Type of Cloudstaff Initiated"] + [c for c in df.columns if c.startswith("Type of ") and c not in
              ("Type of Voluntary Reason", "Type of Involuntary", "Type of Customer Initiated", "Type of Cloudstaff Initiated", "Type of MSA")] + \
             ["Reason for Resignation", "UB Main Reason", "Disengagement Category", "UB Type", "Vol/Inv", "Master Reason"]
BF_COLS = ["Year Disengagement", "Month Disengagement", "Date Disengagement", CONFIG["id_col"],
           "Vol/Inv_backfilled", "UB Type_backfill", "Master Reason_backfill", "main_cat", "sub_cat", "detail_cat",
           "UB Type_backfill_basis", "label_source", "label_evidence"] + FIELD_COLS
out = df.rename(columns={"main_category": "main_cat", "primary_reason_id": "sub_cat", "detail_category": "detail_cat",
                         "classification_source": "label_source", "classification_evidence": "label_evidence"})
out["label_source"] = out["label_source"].replace({"llm": "note", "form fields": "form fields", "rule: blank note": "note (blank)"})
out = out[[c for c in BF_COLS if c in out.columns]].copy()
for c in out.columns:
    if out[c].dtype == object or str(out[c].dtype).startswith("string"):
        out[c] = out[c].astype("string").str.replace("_x000D_", "", regex=False)
BF_XLSX = Path(CONFIG["input_path"]).with_name(f"{Path(CONFIG['input_path']).stem} - backfill to {CONFIG['date_max'][:7]}.xlsx")
cat_defs = category_definitions(out)
_rt = []
for _d, (_leaf, _basis) in BACKFILL_LEAF.items():
    _lab = leaf_label(_leaf)
    _sub = next((r_.sub_cat for r_ in cat_defs.itertuples() if r_.detail_cat == _d), None)
    if _lab is None: _rt.append((_d, _leaf, _basis, "not a tree value"))
    elif _basis in ("exact", "branch only") and _lab[2] != _d: _rt.append((_d, _leaf, _basis, f"maps back to {_lab[2]}"))
    elif _sub and MAIN_MAP.get(_lab[0]) != MAIN_MAP.get(_sub): _rt.append((_d, _leaf, _basis, f"maps back to main {MAIN_MAP.get(_lab[0])}"))
assert not _rt, f"BACKFILL_LEAF defaults that do not map back: {_rt}"
_miss = sorted(set(DETAIL_DEFINITIONS) ^ set(cat_defs["detail_cat"]))
assert not _miss, f"detail_cat definitions and the category list differ: {_miss}"
print(f"Category definitions: {len(cat_defs)} detail rows; every exact / branch-only default maps back to its own detail_cat, every default keeps its main_cat")
about = pd.DataFrame([
    ("Scope", f"Tickets with {CONFIG['date_col']} <= {CONFIG['date_max']} ({len(out):,} rows). Built by the notebook, Phase 11."),
    ("main_cat / sub_cat / detail_cat", "The categorization. The note's reason wins; the form fields are used only where the note gives no reason."),
    ("label_source", "note = from the note text; form fields = note gave no reason, read from the form tree (step 7b)."),
    ("UB Type_backfill", "The form-tree value (written like the sheet's UB Type) that matches the label. '-' = For Record / no reason / Deleted."),
    ("UB Type_backfill_basis", "form (UB Type agrees) / form (path) = taken from the ticket's own form fields; exact = the only tree value for that detail_cat; "
                               "nearest = closest tree value (several fit, or none exactly); branch only = the reason has no sub-reason level."),
    ("Master Reason_backfill", "= main_cat, except For Record is written 'For Record Purpose Only' as in the sheet's Master Reason."),
    ("Vol/Inv_backfilled", "From main_cat: Resignation -> Voluntary; Service Dissatisfaction / Business Reason -> Involuntary; "
                           "For Record -> For Record Purpose Only; Deleted -> Deleted; no reason -> '-'."),
    ("Category definitions", "Sheet with every main_cat -> sub_cat -> detail_cat, their definitions, the default UB Type_backfill for each detail_cat and the tickets in this file."),
    ("Original columns", "Columns after label_evidence are the note and the form fields as they are in the source, incl. the sheet's own UB Type, Vol/Inv and Master Reason."),
], columns=["column", "meaning"])
with pd.ExcelWriter(BF_XLSX, engine="xlsxwriter") as xw:
    out.to_excel(xw, sheet_name="Backfill", index=False)
    cat_defs.to_excel(xw, sheet_name="Category definitions", index=False); about.to_excel(xw, sheet_name="About", index=False)
    cs = xw.sheets["Category definitions"]; wrapf = xw.book.add_format({"text_wrap": True, "valign": "top"})
    for j, w in enumerate([22, 45, 30, 70, 34, 70, 40, 16, 10]): cs.set_column(j, j, w, wrapf)
    cs.freeze_panes(1, 0); cs.autofilter(0, 0, len(cat_defs), len(cat_defs.columns) - 1)
    ws = xw.sheets["Backfill"]; wb = xw.book
    hi = wb.add_format({"bold": True, "bg_color": "#DDEBF7", "border": 1, "text_wrap": True, "valign": "top"})
    for i, c in enumerate(out.columns):
        ws.write(0, i, c, hi if i in range(4, 13) else wb.add_format({"bold": True, "border": 1, "text_wrap": True, "valign": "top"}))
        ws.set_column(i, i, 12 if i < 4 else 24 if i < 13 else 40 if c == TEXT else 18)
    ws.freeze_panes(1, 4); ws.autofilter(0, 0, len(out), len(out.columns) - 1)
    xw.sheets["About"].set_column(0, 0, 28); xw.sheets["About"].set_column(1, 1, 140)
print(f"Backfill written: {BF_XLSX} ({len(out):,} rows, {len(out.columns)} columns)")
RUN["status"]["phase11"] = f"backfill: {BF_XLSX.name}"


## Checks

These assertions stand in for a test suite, so the notebook stays self-contained. They cover:
* blank detection, text normalization and duplicate-note handling;
* mapping unique notes and clusters back to tickets;
* schema validation of the taxonomy and of classification items;
* primary/secondary consistency and UNKNOWN vs BLANK;
* Excel completeness.

In [ ]:
checks = []
def check(label, cond):
    checks.append((label, "PASS" if cond else "FAIL"))

# preprocessing
check("blank: None / NaN / whitespace / placeholder -> BLANK",
      all(processing_status(clean_note(x)) == "BLANK" for x in [None, np.nan, "   ", "{{FIELD_REASON_FOR_DISENGAGEMENT:}}", "`"]))
check("blank: real text -> VALID_TEXT", processing_status(clean_note("Business reason")) == "VALID_TEXT")
check("normalize: _x000D_, whitespace, curly quotes, html", clean_note("Client’s  <b>note</b>_x000D_\n end &amp; more") == "Client's note end & more")
check("normalize: negations kept", clean_note("Not happy, no longer required") == "Not happy, no longer required")
check("original note column untouched", df[TEXT].equals(load_input(CONFIG)[TEXT]))
# duplicates and mapping
check("identical cleaned notes share one note_key", df[df["processing_status"] == "VALID_TEXT"].groupby("note_clean")["note_key"].nunique().max() == 1)
check("unique notes map back to every non-blank ticket", unique_notes["records"].sum() == (df["processing_status"] == "VALID_TEXT").sum())
check("every non-blank ticket has a cluster id; blanks have none",
      df.loc[df["processing_status"] == "VALID_TEXT", "cluster_id"].notna().all() and df.loc[df["processing_status"] == "BLANK", "cluster_id"].isna().all())
check("record ids unique", df["record_id"].is_unique)
# taxonomy / classification schemas
try:
    jsonschema.validate({"categories": [{"category_id": "bad id"}], "open_questions": []}, CONSOLIDATION_SCHEMA); bad_tax_rejected = False
except jsonschema.ValidationError:
    bad_tax_rejected = True
check("taxonomy schema rejects an invalid category", bad_tax_rejected)
good = {"note_key": unique_notes["note_key"].iloc[0], "primary_reason": "UNKNOWN", "primary_subcategory": None, "secondary_reasons": ["UNKNOWN"],
        "classification_type": "UNKNOWN", "evidence": "", "confidence": 0.5, "review_required": False, "review_reason": None}
it, prob = check_item(good, set(unique_notes["note_key"]))
check("classification: secondary never repeats primary", prob is None and it["secondary_reasons"] == [])
check("classification: unknown category id rejected", check_item(good | {"primary_reason": "MADE_UP"}, set(unique_notes["note_key"]))[1] is not None)
check("classification: low confidence forces review", apply_review_rules(good)[0])
if complete:
    nb = df[df["processing_status"] == "VALID_TEXT"]
    check("every ticket classified", df["primary_reason_id"].notna().all())
    check("all primary ids are approved categories", df["primary_reason_id"].isin(CAT_IDS).all())
    check("UNKNOWN vs BLANK: preprocessing blanks are BLANK or resolved from the form fields, never UNKNOWN",
          (df.loc[df["processing_status"] == "BLANK", "primary_reason_id"].eq("BLANK")
           | df.loc[df["processing_status"] == "BLANK", "classification_source"].isin(["form fields", "rule: cancelled ticket"])).all())
    check("primary never listed in its own secondaries",
          not any(p in split_ids(s) for p, s in zip(df["primary_reason_id"], df["secondary_reason_ids"].fillna(""))))
    check("confidence within [0, 1] (form-field tickets have none)",
          df.loc[df["classification_source"] != "form fields", "classification_confidence"].between(0, 1).all())
# excel
if WB.exists():
    xl = pd.ExcelFile(WB)
    need = {"Classified Tickets", "Approved Taxonomy", "Cluster Analysis", "Summary Dashboard", "Validation", "Run Metadata"}
    check("workbook has the required sheets", need <= set(xl.sheet_names))
    ct = pd.read_excel(xl, "Classified Tickets", usecols=["record_id"])
    check("Classified Tickets has one row per ticket", len(ct) == len(df) and ct["record_id"].astype(str).is_unique)
    check("all original columns present", set(load_input(CONFIG).columns) <= set(pd.read_excel(xl, "Classified Tickets", nrows=0).columns))
res_checks = pd.DataFrame(checks, columns=["check", "result"]); display(res_checks)
assert (res_checks["result"] == "PASS").all(), "Some checks failed"
logger.info("Run finished: " + json.dumps(RUN["status"]))